# HUẤN LUYỆN TINH CHỈNH TPS-INBETWEEN PHIÊN BẢN V3.0 (ASYMMETRIC STROKE & OCCLUSION-AWARE GENERATION - 540P SAFE MEMORY)
## ĐỀ TÀI: NGHIÊN CỨU & TINH CHỈNH MÔ HÌNH TPS-INBETWEEN CHO HOẠT HỌA ANIME 2D (PBL6)

> **Mục tiêu & Đột Phá Kỹ Thuật Cốt Lõi Phiên Bản V3.0 (Official Production 14-Cell Interactive Release):**
> 1. **Kiến Trúc Hiển Thị Kép Bất Đối Xứng (Cell 5):** Phá bỏ hoàn toàn ràng buộc pha trộn tuyến tính $V_0 + V_1 = 1.0$. Trang bị `AsymmetricStrokeFusionHead_v3` xuất ra 2 cổng hiển thị độc lập $(V_0, V_1) \in [0, 1]^2$ kết hợp `AsymmetricFeatureAnalyzer` và `ContourGuidedFlowPropagator` để tự động bảo toàn 100% độ đậm nét cho các chi tiết đơn nguồn (mắt, miệng, tai, nếp áo) khi nhân vật quay đầu ($A \to B$ và $B \to A$).
> 2. **Khởi Tạo Kaiming Phá Bẫy Đối Xứng & 2-Stage Curriculum (Cell 5 & 6):** Khởi tạo Kaiming Normal ($a=0.25$) cho các lớp PReLU Conv và Small Normal ($\sigma=10^{-3}$) cho các lớp xuất cổng/dòng, phá bỏ triệt để bẫy 0-gradient ban đầu. Lịch trình 2 giai đoạn: Epoch 1–2 đóng băng backbone + warmup êm các nhánh V3 mới; Epoch 3–25 mở toàn mạng kết hợp Cosine Decay.
> 3. **Bảo Toàn Đỉnh Ngọn Tóc & Chuẩn Hóa Mật Độ Nét Chuẩn Vàng 1.10x (Cell 6):**
>    - `HairTipEndpointContinuityLoss` ($\lambda = 0.30$) với cơ chế dò tìm điểm đầu mút trên khung xương nét vẽ (`skel_gt`), kích hoạt trọng số focal $4.0\times$ để triệt tiêu hoàn toàn đứt nét ở đỉnh các lọn tóc mảnh.
>    - `StrokeDensityRegularizationLoss_v3` với dải Dead-zone chuẩn $[1.04, 1.14]$ (tâm chuẩn lý tưởng **$1.10\times$**), kết hợp Exact-STE tại ngưỡng $0.15$ và nhánh Leaky Gradient.
>    - `OcclusionAwareGhostLoss` tích hợp Background Cleanliness Penalty ($0.35 \cdot \text{ReLU}(p_{\text{f}} - 0.008)$) triệt tiêu màng sương và bóng ma xa.
> 4. **An Toàn Tuyệt Đối VRAM & Host RAM Trên Tập Gốc 540p (Cell 1, 3, 4, 10):**
>    - Huấn luyện trên tập gốc 540p (`sakuga_golden_dataset_tps_ready_v1_patch3`, 8,554 triplets, $540 \times 960$) với Random Crop $512 \times 512$.
>    - `BATCH_SIZE = 8` (4 mẫu/GPU trên 2x T4), `GRAD_ACCUM_STEPS = 2` $\implies$ Effective Batch Size = 16, VRAM $< 9.2\text{ GB}$/GPU (tránh OOM trên GPU 0 khi tính VGG + clDice).
>    - Tiền tính toán cache TPS dạng `.npy` uncompressed kết hợp `np.load(..., mmap_mode='r')` (0 heap RAM, zero Copy-on-Write page duplication giữa DataLoader workers).
>    - Tự động giải phóng Linux PageCache qua `posix_fadvise(POSIX_FADV_DONTNEED)` khi đọc ảnh và dọn dẹp RAM vật lý (`cleanup_memory()`, `malloc_trim(0)`) sau mỗi epoch.
> 5. **Validation Thuần FP32 (Cell 7, 8, 10):** Tắt hoàn toàn autocast FP16 khi thẩm định (`use_amp=False`) với `torch.inference_mode()`, unpad từ 544 về đúng độ phân giải gốc $540 \times 960$, đo lường $100\%$ chỉ số trên GPU không làm sai lệch mật độ nét.


In [ ]:
# [CELL 1]: Cấu hình Môi trường, Phần cứng Kaggle (2x T4 GPU) & Trình Dọn Rác Bộ Nhớ Chủ Động
import os
import sys
import ctypes

os.environ["MALLOC_TRIM_THRESHOLD_"] = "65536"
os.environ["MALLOC_ARENA_MAX"] = "2"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
# [AN TOÀN MULTI-GPU]: Không bật expandable_segments:True khi dùng nn.DataParallel trên 2x T4
# để tránh lỗi cuMemMap / illegal memory access khi cấp phát chéo stream giữa cuda:0 và cuda:1

# Tắt monitor thread của tqdm để tránh xung đột luồng IOPub của Jupyter làm treo "Run All"
import tqdm
tqdm.monitor_interval = 0

import gc
import time
import math
import random
import json
import shutil
import numpy as np
import pandas as pd
from PIL import Image
import cv2
cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.multiprocessing as mp
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from torch.utils.data.dataloader import default_collate
import torchvision.models as models

# [CHIẾN LƯỢC CHIA SẺ TENSOR GIỮA WORKER]: 'file_descriptor' (mặc định PyTorch trên Linux).
# Lưu ý: 'file_system' KHÔNG ghi xuống ổ đĩa - nó vẫn tạo segment trong /dev/shm (RAM tmpfs) và
# chỉ được dọn khi torch_shm_manager thu gom => dễ để lại rác. 'file_descriptor' shm_unlink ngay,
# kernel tự thu hồi khi fd đóng.
try:
    mp.set_sharing_strategy('file_descriptor')
except Exception:
    pass

# Nâng giới hạn số file descriptor (file_descriptor strategy cần nhiều fd)
try:
    import resource
    _soft, _hard = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (_hard, _hard))
except Exception:
    pass

def configure_glibc_allocator():
    try:
        libc = ctypes.CDLL('libc.so.6')
        libc.mallopt(-1, 65536)
        libc.mallopt(-8, 2)
    except Exception:
        pass

configure_glibc_allocator()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# [CỰC KỲ QUAN TRỌNG TRÊN 2x T4]: Tắt cudnn.benchmark để ngăn xung đột cuDNN autotuner đa luồng
torch.backends.cudnn.benchmark = False

NUM_GPUS = torch.cuda.device_count()
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print("=" * 80)
print(f"[*] THIẾT LẬP MÔI TRƯỜNG & PHẦN CỨNG HUẤN LUYỆN V3.0 (14 CELLS)")
print(f"[*] Thiết bị chính: {device} | Số lượng GPU khả dụng: {NUM_GPUS}")
for i in range(NUM_GPUS):
    prop = torch.cuda.get_device_properties(i)
    print(f"    - GPU {i}: {prop.name} (VRAM: {prop.total_memory / 1e9:.2f} GB)")

# Ngưỡng watchdog RAM (working set container). Vượt ngưỡng => lưu resume & dừng êm.
RAM_SOFT_LIMIT_GB = 28.0
RAM_WARN_GB = 24.0

def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    try:
        ctypes.CDLL('libc.so.6').malloc_trim(0)
    except Exception:
        pass

def _read_cgroup_memory_bytes():
    for p in ('/sys/fs/cgroup/memory.current', '/sys/fs/cgroup/memory/memory.usage_in_bytes'):
        try:
            with open(p, 'r') as f:
                return int(f.read().strip())
        except Exception:
            continue
    return None

def _read_cgroup_memory_stat():
    """Đọc memory.stat (cgroup v2 hoặc v1) -> dict {anon, file, shmem, active_file, inactive_file} (bytes)."""
    out = {}
    for p, mapping in (
        ('/sys/fs/cgroup/memory.stat', {'anon': 'anon', 'file': 'file', 'shmem': 'shmem',
                                        'active_file': 'active_file', 'inactive_file': 'inactive_file'}),
        ('/sys/fs/cgroup/memory/memory.stat', {'total_rss': 'anon', 'total_cache': 'file', 'total_shmem': 'shmem',
                                               'total_active_file': 'active_file', 'total_inactive_file': 'inactive_file'}),
    ):
        try:
            with open(p, 'r') as f:
                for line in f:
                    parts = line.split()
                    if len(parts) == 2 and parts[0] in mapping:
                        out[mapping[parts[0]]] = int(parts[1])
            if out:
                return out
        except Exception:
            continue
    return out

def _read_pss_bytes(pid):
    try:
        with open(f'/proc/{pid}/smaps_rollup', 'r') as f:
            for line in f:
                if line.startswith('Pss:'):
                    return int(line.split()[1]) * 1024
    except Exception:
        pass
    return 0

def get_memory_stats():
    """Thống kê RAM chính xác (không cộng trùng RSS):
    working_set (= thanh RAM Kaggle), anon, file (PageCache), shmem, /dev/shm, PSS tổng, số tiến trình con."""
    G = 1024 ** 3
    s = {'cgroup': 0.0, 'working_set': 0.0, 'anon': 0.0, 'file': 0.0, 'shmem': 0.0,
         'devshm': 0.0, 'pss': 0.0, 'children': 0}
    cg = _read_cgroup_memory_bytes()
    st = _read_cgroup_memory_stat()
    if cg is not None:
        s['cgroup'] = cg / G
        s['working_set'] = max(0, cg - st.get('inactive_file', 0)) / G
    for k in ('anon', 'file', 'shmem'):
        if k in st:
            s[k] = st[k] / G
    try:
        if os.path.exists('/dev/shm'):
            s['devshm'] = shutil.disk_usage('/dev/shm').used / G
    except Exception:
        pass
    try:
        import psutil
        proc = psutil.Process(os.getpid())
        children = proc.children(recursive=True)
        s['children'] = len(children)
        pss = _read_pss_bytes(proc.pid) + sum(_read_pss_bytes(c.pid) for c in children)
        if pss == 0:
            pss = proc.memory_info().rss  # fallback (Windows/không có smaps_rollup)
        s['pss'] = pss / G
        if cg is None:
            s['working_set'] = s['pss']
    except Exception:
        pass
    return s

def get_host_ram_gb():
    return get_memory_stats()['working_set']

def format_host_ram(s=None):
    s = s or get_memory_stats()
    return (f"WS={s['working_set']:.2f} GB | anon={s['anon']:.2f} | file={s['file']:.2f} | "
            f"shmem={s['shmem']:.2f} | /dev/shm={s['devshm']:.2f} | PSS={s['pss']:.2f} | children={s['children']}")

def drop_file_page_cache(path_or_fd):
    """Giải phóng các trang file đơn lẻ khỏi Linux OS Page Cache (TUYỆT ĐỐI không walk thư mục FUSE)."""
    if not hasattr(os, 'posix_fadvise') or not hasattr(os, 'POSIX_FADV_DONTNEED'):
        return
    try:
        if isinstance(path_or_fd, int):
            os.posix_fadvise(path_or_fd, 0, 0, os.POSIX_FADV_DONTNEED)
        elif isinstance(path_or_fd, str) and os.path.exists(path_or_fd) and os.path.isfile(path_or_fd):
            fd = os.open(path_or_fd, os.O_RDONLY)
            try:
                os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED)
            finally:
                os.close(fd)
    except Exception:
        pass

_HAS_FADVISE = hasattr(os, 'posix_fadvise') and hasattr(os, 'POSIX_FADV_DONTNEED')

def imread_gray(path):
    """Đọc ảnh xám: os.read -> cv2.imdecode, sau đó POSIX_FADV_DONTNEED để bỏ file khỏi PageCache ngay
    (chống PageCache phình theo epoch khi WeightedRandomSampler chạm dần file mới)."""
    if not path:
        return None
    try:
        fd = os.open(path, os.O_RDONLY)
        try:
            chunks = []
            while True:
                b = os.read(fd, 1 << 20)
                if not b:
                    break
                chunks.append(b)
            buf = b''.join(chunks)
            if _HAS_FADVISE:
                try:
                    os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED)
                except Exception:
                    pass
        finally:
            os.close(fd)
        im = cv2.imdecode(np.frombuffer(buf, dtype=np.uint8), cv2.IMREAD_GRAYSCALE)
        del buf, chunks
        if im is not None:
            return im
    except Exception:
        pass
    try:
        with Image.open(path) as img:
            return np.array(img.convert('L'), dtype=np.uint8)
    except Exception:
        return None

WORKING_DIR = '/kaggle/working' if os.path.exists('/kaggle') else './workspace'
os.makedirs(WORKING_DIR, exist_ok=True)
CACHE_DIR = os.path.join(WORKING_DIR, 'dataset_cache')
os.makedirs(CACHE_DIR, exist_ok=True)
CKPT_DIR = os.path.join(WORKING_DIR, 'checkpoints_v3.0')
os.makedirs(CKPT_DIR, exist_ok=True)

print(f"[*] Thư mục làm việc: {WORKING_DIR}")
print(f"[*] RAM Host ban đầu : {format_host_ram()}")
print(f"[*] Watchdog RAM     : cảnh báo > {RAM_WARN_GB:.0f} GB | lưu resume & dừng êm > {RAM_SOFT_LIMIT_GB:.0f} GB")
print("=" * 80)
sys.stdout.flush()


In [ ]:
# [CELL 2]: Tự Động Định Vị, Giải Nén Siêu Tốc & Thẩm Định Toàn Vẹn Golden Dataset 540p (Patch 3 Ready)
import os
import sys
import gc
import time
import zipfile
import tarfile
import subprocess
import glob
import pandas as pd

KAGGLE_INPUT = '/kaggle/input' if os.path.exists('/kaggle/input') else './data'

TMP_DIR = os.path.join(WORKING_DIR, '_temp_dataset')
EXTRACT_DIR = os.path.join(TMP_DIR, 'sakuga_golden_dataset_tps_ready')
DATASET_EXTRACTED_TO_TEMP = False

DATASET_ROOT = None
SKIP_DATA_SUBDIRS = {'train', 'val', 'test', 'matches', 'frames', 'clips', 'crops', 'images', 'npy', 'wandb', '.git', '__pycache__', 'cache'}

search_locations = [KAGGLE_INPUT, EXTRACT_DIR, TMP_DIR, WORKING_DIR, './data/processed', '../data/processed', '.']

t_find_start = time.time()
print("=" * 80)
print("[CELL 2] 🔍 [Bước 1/3] QUÉT ĐỊNH VỊ SIÊU TỐC GOLDEN DATASET 540P (V1_PATCH3)...")
print("=" * 80)

# Ưu tiên kiểm tra trực tiếp các đường dẫn cố định chứa patch3
direct_dataset_candidates = [
    os.path.join(KAGGLE_INPUT, 'sakuga_golden_dataset_tps_ready_v1_patch3', 'anime_tps_finetune_dataset_ready'),
    os.path.join(KAGGLE_INPUT, 'sakuga_golden_dataset_tps_ready_v1_patch3'),
    os.path.join(KAGGLE_INPUT, 'sakuga-golden-dataset-tps-ready-v1-patch3', 'anime_tps_finetune_dataset_ready'),
    os.path.join(KAGGLE_INPUT, 'sakuga-golden-dataset-tps-ready-v1-patch3'),
    './data/processed/sakuga_golden_dataset_tps_ready_v1_patch3/anime_tps_finetune_dataset_ready',
    './data/processed/sakuga_golden_dataset_tps_ready_v1_patch3',
    EXTRACT_DIR,
    os.path.join(TMP_DIR, 'anime_tps_finetune_dataset_ready'),
    './data/processed/anime_tps_finetune_dataset_ready',
    './data/processed'
]

shallow_patterns = [
    os.path.join(KAGGLE_INPUT, 'datasets', '*', '*', 'anime_tps_finetune_dataset_ready'),
    os.path.join(KAGGLE_INPUT, '*', 'anime_tps_finetune_dataset_ready'),
    os.path.join(KAGGLE_INPUT, 'datasets', '*', '*'),
    os.path.join(KAGGLE_INPUT, '*')
]
for pat in shallow_patterns:
    for m_dir in glob.glob(pat, recursive=False):
        if m_dir not in direct_dataset_candidates:
            direct_dataset_candidates.append(m_dir)

for cand in direct_dataset_candidates:
    if os.path.exists(cand) and os.path.isdir(cand):
        if os.path.exists(os.path.join(cand, 'triplets.csv')) or os.path.exists(os.path.join(cand, 'triplets_combined.csv')):
            DATASET_ROOT = cand
            print(f"    ⚡ Định vị trực tiếp thành công tại: {DATASET_ROOT}")
            break

if DATASET_ROOT is None:
    print("    [*] Quét mở rộng tìm thư mục metadata...")
    for base_dir in search_locations:
        if not os.path.exists(base_dir):
            continue
        for root, dirs, files in os.walk(base_dir):
            dirs[:] = [d for d in dirs if d.lower() not in SKIP_DATA_SUBDIRS and not d.startswith('.') and not any(k in d.lower() for k in ['crop', 'clip', 'frame'])]
            if 'triplets.csv' in files or 'triplets_combined.csv' in files:
                if DATASET_ROOT is None or "patch3" in root.lower() or "v1_patch3" in root.lower():
                    DATASET_ROOT = root
                    if "patch3" in root.lower() or "v1_patch3" in root.lower():
                        break
        if DATASET_ROOT is not None and ("patch3" in DATASET_ROOT.lower() or "v1_patch3" in DATASET_ROOT.lower()):
            break

# Bước 2: Quét archive (.zip / .tar.gz) nếu chưa có thư mục giải nén
if DATASET_ROOT is None:
    print("[*] Quét tìm file nén (.zip / .tar.gz) chứa dataset...")
    archive_path = None
    archive_type = None
    valid_keywords = ['patch3', 'patch_3', 'v1_patch3', 'sakuga', 'golden', 'tps']

    for base_dir in search_locations:
        if not os.path.exists(base_dir):
            continue
        for root, dirs, files in os.walk(base_dir):
            dirs[:] = [d for d in dirs if d.lower() not in SKIP_DATA_SUBDIRS and not d.startswith('.') and not any(k in d.lower() for k in ['crop', 'clip', 'frame'])]
            for f in files:
                f_lower = f.lower()
                is_match = any(k in f_lower for k in valid_keywords)
                if is_match and (f_lower.endswith('.tar.gz') or f_lower.endswith('.tgz')):
                    archive_path = os.path.join(root, f)
                    archive_type = 'tar'
                    if "patch3" in f_lower: break
                elif is_match and f_lower.endswith('.zip'):
                    archive_path = os.path.join(root, f)
                    archive_type = 'zip'
                    if "patch3" in f_lower: break
            if archive_path is not None and "patch3" in archive_path.lower(): break
        if archive_path is not None and "patch3" in archive_path.lower(): break

    if archive_path is not None and os.path.exists(archive_path):
        print(f"[✓] Tìm thấy archive: {archive_path}. Tiến hành giải nén vào {EXTRACT_DIR}...")
        os.makedirs(EXTRACT_DIR, exist_ok=True)
        DATASET_EXTRACTED_TO_TEMP = True
        if archive_type == 'tar':
            subprocess.run(['tar', '-xf', archive_path, '-C', EXTRACT_DIR], check=True)
        else:
            with zipfile.ZipFile(archive_path, 'r') as zf:
                zf.extractall(EXTRACT_DIR)
        DATASET_ROOT = EXTRACT_DIR

if DATASET_ROOT is None:
    raise FileNotFoundError("Không thể tìm thấy Sakuga Golden Dataset 540p! Vui lòng kiểm tra Kaggle Input.")

meta_csv = os.path.join(DATASET_ROOT, 'triplets.csv')
if not os.path.exists(meta_csv):
    meta_csv = os.path.join(DATASET_ROOT, 'triplets_combined.csv')

df_meta = pd.read_csv(meta_csv)
print(f"[✓] Đã nạp metadata: {meta_csv} ({len(df_meta):,} mẫu)")

if 'split' in df_meta.columns:
    train_df = df_meta[df_meta['split'] == 'train'].reset_index(drop=True)
    val_df = df_meta[df_meta['split'] == 'val'].reset_index(drop=True)
else:
    n_train = int(len(df_meta) * 0.85)
    train_df = df_meta.iloc[:n_train].reset_index(drop=True)
    val_df = df_meta.iloc[n_train:].reset_index(drop=True)

print(f"    - Mẫu Huấn Luyện (Train): {len(train_df):,}")
print(f"    - Mẫu Thẩm Định (Val)   : {len(val_df):,}")
drop_file_page_cache(meta_csv)
print("=" * 80)
sys.stdout.flush()

In [ ]:
# [CELL 3]: Hệ Thống Biến Dạng Lưới, 8 Điểm Neo Biên/Góc & Vector Hóa TPS Flow Batch
import os
import sys
import math
import threading
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

_GRID_LOCK = threading.Lock()
_NORM_GRID_CACHE = {}
_PIXEL_GRID_CACHE = {}
_MAX_GRID_CACHE_SIZE = 32

def warp_normalized_flow(img, flow):
    """
    Biến dạng lưới Coarse TPS (tọa độ chuẩn hóa [0, 1] sang [-1, 1]).
    Caching lưới cơ sở (1, H, W, 2) và cộng trực tiếp qua broadcasting.
    """
    b, _, h, w = img.shape
    dev = flow.device
    dtype = flow.dtype
    key = (h, w, dev, dtype)
    base_grid = _NORM_GRID_CACHE.get(key)
    if base_grid is None:
        with _GRID_LOCK:
            base_grid = _NORM_GRID_CACHE.get(key)
            if base_grid is None:
                if len(_NORM_GRID_CACHE) >= _MAX_GRID_CACHE_SIZE:
                    _NORM_GRID_CACHE.clear()
                grid_y, grid_x = torch.meshgrid(
                    torch.linspace(0, 1, h, device=dev, dtype=dtype),
                    torch.linspace(0, 1, w, device=dev, dtype=dtype), indexing='ij'
                )
                base_grid = torch.stack([grid_x, grid_y], dim=-1).unsqueeze(0).contiguous()
                _NORM_GRID_CACHE[key] = base_grid

    with torch.amp.autocast('cuda', enabled=False):
        flow_f32 = flow.float()
        grid = ((base_grid + flow_f32.permute(0, 2, 3, 1)) * 2.0 - 1.0).contiguous()
        out = F.grid_sample(img.float().contiguous(), grid, padding_mode='border', align_corners=True)
    return out.to(img.dtype)

def warp_pixel_flow(img, flow):
    """
    Biến dạng lưới Fine Optical Flow IFNet (tọa độ pixel thực tế).
    Caching lưới cơ sở [-1, 1] và cộng trực tiếp qua broadcasting.
    """
    b, _, h, w = img.shape
    dev = flow.device
    dtype = flow.dtype
    key = (h, w, dev, dtype)
    base_grid = _PIXEL_GRID_CACHE.get(key)
    if base_grid is None:
        with _GRID_LOCK:
            base_grid = _PIXEL_GRID_CACHE.get(key)
            if base_grid is None:
                if len(_PIXEL_GRID_CACHE) >= _MAX_GRID_CACHE_SIZE:
                    _PIXEL_GRID_CACHE.clear()
                grid_y, grid_x = torch.meshgrid(
                    torch.linspace(-1, 1, h, device=dev, dtype=dtype),
                    torch.linspace(-1, 1, w, device=dev, dtype=dtype), indexing='ij'
                )
                base_grid = torch.stack([grid_x, grid_y], dim=-1).unsqueeze(0).contiguous()
                _PIXEL_GRID_CACHE[key] = base_grid

    with torch.amp.autocast('cuda', enabled=False):
        flow_f32 = flow.float()
        flow_norm = torch.cat([
            flow_f32[:, 0:1, :, :] / max(1.0, (w - 1.0) / 2.0),
            flow_f32[:, 1:2, :, :] / max(1.0, (h - 1.0) / 2.0)
        ], dim=1)
        grid = (base_grid + flow_norm.permute(0, 2, 3, 1)).contiguous()
        out = F.grid_sample(img.float().contiguous(), grid, padding_mode='border', align_corners=True)
    return out.to(img.dtype)

def tps_rbf(r):
    r = torch.clamp(r, min=1e-6)
    return (r ** 2) * torch.log(r)

# 8 Điểm neo tĩnh (4 góc + 4 điểm giữa biên) khóa chặt khung hình, chống co rút viền ảnh
FIXED_ANCHORS_NP = np.array([
    [0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0],
    [0.5, 0.0], [0.5, 1.0], [0.0, 0.5], [1.0, 0.5]
], dtype=np.float32)

def filter_spacing_np(c0, c1, min_dist=0.015, max_kps=80):
    n = c0.shape[0]
    if n <= 1:
        return c0, c1
    disp = np.linalg.norm(c1 - c0, axis=-1)
    order = np.argsort(-disp)
    c0_s, c1_s = c0[order], c1[order]
    keep = [0]
    for i in range(1, n):
        dists = np.linalg.norm(c0_s[keep] - c0_s[i], axis=-1)
        if np.all(dists >= min_dist):
            keep.append(i)
            if len(keep) >= max_kps:
                break
    return c0_s[keep], c1_s[keep]

def filter_topo_np(c_s, c_d, max_dev=0.18, min_neighbors=3, radius=0.25):
    n = c_s.shape[0]
    if n <= 6:
        return np.arange(n)
    disp = c_d - c_s
    diff_s = c_s[:, None, :] - c_s[None, :, :]
    dist_sq = diff_s[:, :, 0] ** 2 + diff_s[:, :, 1] ** 2
    nbr_mask = (dist_sq < radius * radius) & (dist_sq > 1e-10)
    nbr_counts = nbr_mask.sum(axis=1)
    keep = []
    for i in range(n):
        if nbr_counts[i] < min_neighbors:
            keep.append(i)
            continue
        nbr_disp = disp[nbr_mask[i]]
        diff = np.abs(nbr_disp - disp[i])
        dev = np.median(np.sqrt(diff[:, 0] ** 2 + diff[:, 1] ** 2))
        if dev <= max_dev:
            keep.append(i)
    if len(keep) < 4:
        return np.arange(n)
    return np.array(keep, dtype=int)

def solve_tps_single_cpu(c_s, c_d, max_kps=80, lambd=0.5, min_dist=0.015, max_pts=88, return_tensor=False):
    """
    Giải hệ phương trình Thin-Plate Spline đơn mẫu trên CPU tốc độ cao (< 0.4ms).
    """
    if c_s is None or c_d is None or c_s.shape[0] < 4:
        c_pad = np.zeros((max_pts, 2), dtype=np.float32)
        th_pad = np.zeros((max_pts + 3, 2), dtype=np.float32)
        return (torch.from_numpy(c_pad), torch.from_numpy(th_pad), 0.0) if return_tensor else (c_pad, th_pad, 0.0)

    n = c_s.shape[0]
    min_dist_sq = min_dist * min_dist
    keep_indices = [0]
    kept_pts = np.empty((n, 2), dtype=np.float32)
    kept_pts[0] = c_s[0]
    num_kept = 1

    for i in range(1, n):
        dx = kept_pts[:num_kept, 0] - c_s[i, 0]
        dy = kept_pts[:num_kept, 1] - c_s[i, 1]
        if np.min(dx * dx + dy * dy) >= min_dist_sq:
            kept_pts[num_kept] = c_s[i]
            keep_indices.append(i)
            num_kept += 1
            if num_kept >= max_kps:
                break

    if num_kept < 4:
        c_pad = np.zeros((max_pts, 2), dtype=np.float32)
        th_pad = np.zeros((max_pts + 3, 2), dtype=np.float32)
        return (torch.from_numpy(c_pad), torch.from_numpy(th_pad), 0.0) if return_tensor else (c_pad, th_pad, 0.0)

    c_s_filt = kept_pts[:num_kept]
    c_d_filt = c_d[keep_indices]

    c_s_anchors = np.concatenate([c_s_filt, FIXED_ANCHORS_NP], axis=0)
    c_d_anchors = np.concatenate([c_d_filt, FIXED_ANCHORS_NP], axis=0)
    m = c_s_anchors.shape[0]

    diff = c_s_anchors[:, None, :] - c_s_anchors[None, :, :]
    dist = np.sqrt(np.sum(diff * diff, axis=-1))
    dist_clamped = np.maximum(dist, 1e-6)
    K = (dist_clamped ** 2) * np.log(dist_clamped) + lambd * np.eye(m, dtype=np.float32)
    P = np.concatenate([np.ones((m, 1), dtype=np.float32), c_s_anchors], axis=-1)
    L = np.concatenate([np.concatenate([K, P], axis=-1), np.concatenate([P.T, np.zeros((3, 3), dtype=np.float32)], axis=-1)], axis=0)
    Y = np.concatenate([c_d_anchors, np.zeros((3, 2), dtype=np.float32)], axis=0)

    try:
        theta = np.linalg.solve(L, Y)
    except Exception:
        theta = np.linalg.lstsq(L, Y, rcond=None)[0]
    theta = np.nan_to_num(theta, nan=0.0, posinf=0.0, neginf=0.0)
    v_flag = 0.0 if (np.isnan(theta).any() or np.isinf(theta).any() or np.max(np.abs(theta)) > 50.0) else 1.0

    c_pad = np.zeros((max_pts, 2), dtype=np.float32)
    theta_pad = np.zeros((max_pts + 3, 2), dtype=np.float32)
    c_pad[:m] = c_s_anchors
    theta_pad[:m] = theta[:m]
    theta_pad[max_pts:max_pts+3] = theta[m:m+3]

    if return_tensor:
        return torch.from_numpy(c_pad), torch.from_numpy(theta_pad), float(v_flag)
    return c_pad, theta_pad, float(v_flag)

def solve_tps_pair_cpu(kps, max_kps=80, lambd=0.5, min_dist=0.015, return_tensor=True):
    """
    Giải song song 2 chiều (Forward 0->1 và Backward 1->0) cho 1 cặp keypoints.
    """
    MAX_PTS = 88
    if kps is None or len(kps) < 2 or (hasattr(kps, 'shape') and len(kps.shape) > 1 and kps.shape[1] < 4):
        z_c0 = np.zeros((MAX_PTS, 2), dtype=np.float32)
        z_th01 = np.zeros((MAX_PTS + 3, 2), dtype=np.float32)
        z_c1 = np.zeros((MAX_PTS, 2), dtype=np.float32)
        z_th10 = np.zeros((MAX_PTS + 3, 2), dtype=np.float32)
        z_v = np.array(0.0, dtype=np.float32)
        if return_tensor:
            return (torch.from_numpy(z_c0), torch.from_numpy(z_th01), torch.from_numpy(z_c1),
                    torch.from_numpy(z_th10), torch.from_numpy(z_v))
        return z_c0, z_th01, z_c1, z_th10, z_v

    c0 = kps[0]
    c1 = kps[1]
    c0_pad, th01_pad, v01 = solve_tps_single_cpu(c0, c1, max_kps=max_kps, lambd=lambd, min_dist=min_dist, max_pts=MAX_PTS, return_tensor=False)
    c1_pad, th10_pad, v10 = solve_tps_single_cpu(c1, c0, max_kps=max_kps, lambd=lambd, min_dist=min_dist, max_pts=MAX_PTS, return_tensor=False)

    v = np.array(1.0 if (v01 > 0.5 and v10 > 0.5) else 0.0, dtype=np.float32)
    if return_tensor:
        return (torch.from_numpy(c0_pad), torch.from_numpy(th01_pad), torch.from_numpy(c1_pad),
                torch.from_numpy(th10_pad), torch.from_numpy(v))
    return c0_pad, th01_pad, c1_pad, th10_pad, v

_GRID_CACHE = {}

def get_grid_pts(h, w, device):
    key = (h, w, device)
    pts = _GRID_CACHE.get(key)
    if pts is None:
        if len(_GRID_CACHE) >= _MAX_GRID_CACHE_SIZE:
            _GRID_CACHE.clear()
        grid_y, grid_x = torch.meshgrid(
            torch.linspace(0, 1, h, device=device),
            torch.linspace(0, 1, w, device=device), indexing='ij'
        )
        pts = torch.stack([grid_x, grid_y], dim=-1).view(-1, 2)
        _GRID_CACHE[key] = pts
    return pts

@torch.no_grad()
def tps_flow_from_theta_batch(theta_batch, c_src_batch, valid_mask, h, w, device, chunk_size=65536):
    """
    Vector hóa 100% trên GPU: Tính toán trường dòng chuyển động TPS cho TOÀN BỘ BATCH cùng lúc.
    """
    with torch.amp.autocast('cuda', enabled=False):
        theta_f32 = theta_batch.float()
        c_src_f32 = c_src_batch.float()
        valid_f32 = valid_mask.float()

        b = theta_f32.shape[0]
        pts = get_grid_pts(h, w, device)
        total_pts = pts.shape[0]
        flow_chunks = []

        for i in range(0, total_pts, chunk_size):
            p_chunk = pts[i:i + chunk_size]
            c_len = p_chunk.shape[0]
            pts_exp = p_chunk.unsqueeze(0).expand(b, -1, -1)

            dist = torch.cdist(pts_exp, c_src_f32, p=2.0)
            U = tps_rbf(dist)

            ones = torch.ones((b, c_len, 1), device=device, dtype=pts.dtype)
            P = torch.cat([ones, pts_exp], dim=-1)
            basis = torch.cat([U, P], dim=-1)

            mapped = torch.bmm(basis, theta_f32)
            flow_chunks.append(mapped - pts_exp)

        flow = torch.cat(flow_chunks, dim=1)
        flow = flow.view(b, h, w, 2).permute(0, 3, 1, 2).contiguous()
        flow.mul_(valid_f32.view(b, 1, 1, 1))
        return flow

def tps_flow_from_theta(theta, c_src, h, w, device, chunk_size=32768):
    """Fallback tương thích cho đơn mẫu inference."""
    n = c_src.shape[0]
    if n < 4 or theta.shape[0] != (n + 3):
        return torch.zeros((1, 2, h, w), device=device)

    pts = get_grid_pts(h, w, device)
    total_pts = pts.shape[0]
    flow_chunks = []

    for i in range(0, total_pts, chunk_size):
        p_chunk = pts[i:i + chunk_size]
        diff = p_chunk.unsqueeze(1) - c_src.unsqueeze(0)
        dist = torch.norm(diff, dim=-1)
        U = tps_rbf(dist)
        P = torch.cat([torch.ones((p_chunk.shape[0], 1), device=device), p_chunk], dim=-1)
        basis = torch.cat([U, P], dim=-1)
        mapped = torch.matmul(basis, theta)
        flow_chunk = mapped - p_chunk
        flow_chunks.append(flow_chunk)

    flow = torch.cat(flow_chunks, dim=0)
    flow = flow.view(h, w, 2).permute(2, 0, 1).unsqueeze(0)
    return flow

print("=" * 80)
print("[✓] CELL 3 hoàn tất: Warp Layers Cache, 8 Anchors & Vectorized TPS Batch Solver sẵn sàng (< 0.05s).")
print("=" * 80)
sys.stdout.flush()


In [ ]:
# [CELL 4]: Lớp Dataset SakugaGoldenDataset_v3 (.npy mmap 0-copy) & DataLoader Huấn Luyện 540p Toàn Phần
import os
import sys
import gc
import random
import time
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.utils.data.dataloader import default_collate

class TemporalReversalAugmentation:
    def __init__(self, p=0.5):
        self.p = p

    def __call__(self, img0, imggt, img1, t_val, kps=None,
                 c0_pad=None, th01_pad=None, c1_pad=None, th10_pad=None):
        if random.random() < self.p:
            img0_out, img1_out = img1, img0
            t_out = float(1.0 - t_val)
            gt_out = imggt
            kps_out = np.stack([kps[1], kps[0]], axis=0) if (kps is not None and kps.shape[0] == 2) else kps
            c0_out, th01_out = c1_pad, th10_pad
            c1_out, th10_out = c0_pad, th01_pad
            return img0_out, gt_out, img1_out, t_out, kps_out, c0_out, th01_out, c1_out, th10_out, True
        return img0, imggt, img1, float(t_val), kps, c0_pad, th01_pad, c1_pad, th10_pad, False

def compute_v3_curriculum_weights(df):
    weights = np.ones(len(df), dtype=np.float32)
    t_vals = df["t"].values.astype(np.float32)
    disp_vals = df["mean_disp"].values.astype(np.float32) if "mean_disp" in df.columns else np.zeros(len(df), dtype=np.float32)
    occ_scores = df["patch_occ_score"].values.astype(np.float32) if "patch_occ_score" in df.columns else np.zeros(len(df), dtype=np.float32)
    ink_diffs = df["global_ink_diff"].values.astype(np.float32) if "global_ink_diff" in df.columns else np.zeros(len(df), dtype=np.float32)

    is_high_occ = (occ_scores >= 0.08)
    weights[is_high_occ] *= 1.45

    is_ink_diverge = (ink_diffs >= 0.15)
    weights[is_ink_diverge] *= 1.25

    is_boundary_t = ((t_vals >= 0.15) & (t_vals <= 0.30)) | ((t_vals >= 0.70) & (t_vals <= 0.85))
    is_large_disp = (disp_vals >= 7.0)
    weights[is_boundary_t] *= 1.20
    weights[is_large_disp] *= 1.25

    return weights

class SakugaGoldenDataset_v3(Dataset):
    def __init__(self, root_dir, df_split, is_train=True, crop_size=None, cache_dir=None):
        self.root_dir = root_dir
        self.is_train = is_train
        self.crop_size = crop_size
        self.split_name = 'train' if is_train else 'val'
        self.length = len(df_split)
        self.temporal_aug = TemporalReversalAugmentation(p=0.5) if is_train else None

        # 1. Định vị đường dẫn gốc 1 lần duy nhất (< 0.005s)
        sample_f0 = str(df_split['frame0'].iloc[0]).replace('\\', '/').lstrip('/')
        actual_root = root_dir
        for cr in [root_dir, os.path.join(root_dir, 'anime_tps_finetune_dataset_ready'), os.path.join(root_dir, 'sakuga_golden_dataset_tps_ready_v1_patch3')]:
            if os.path.exists(os.path.join(cr, sample_f0)):
                actual_root = cr
                break

        sample_fm = str(df_split['match_file'].iloc[0]).replace('\\', '/').lstrip('/')
        actual_match_root = actual_root
        for cr in [actual_root, root_dir, os.path.join(root_dir, 'matches')]:
            if os.path.exists(os.path.join(cr, sample_fm)):
                actual_match_root = cr
                break
            if os.path.exists(os.path.join(cr, 'matches', sample_fm)):
                actual_match_root = os.path.join(cr, 'matches')
                break

        f0_list = [os.path.join(actual_root, str(f).replace('\\', '/').lstrip('/')) for f in df_split['frame0']]
        fgt_list = [os.path.join(actual_root, str(f).replace('\\', '/').lstrip('/')) for f in df_split['frame_gt']]
        f1_list = [os.path.join(actual_root, str(f).replace('\\', '/').lstrip('/')) for f in df_split['frame1']]
        fm_list = [os.path.join(actual_match_root, str(f).replace('\\', '/').lstrip('/')) for f in df_split['match_file']]

        self.p0_arr = np.array([f.encode('utf-8') for f in f0_list], dtype='S256')
        self.pgt_arr = np.array([f.encode('utf-8') for f in fgt_list], dtype='S256')
        self.p1_arr = np.array([f.encode('utf-8') for f in f1_list], dtype='S256')
        self.pmatch_arr = np.array([f.encode('utf-8') for f in fm_list], dtype='S256')
        self.t_arr = df_split['t'].to_numpy(dtype=np.float32, copy=True)
        self.clips_arr = np.array([str(c).encode('utf-8') for c in df_split['clip_id']], dtype='S64')

        # 2. Quản lý Cache Keypoints .npy uncompressed (mmap 0-copy)
        cd = cache_dir or CACHE_DIR
        os.makedirs(cd, exist_ok=True)
        kps_npy = os.path.join(cd, f'kps_data_{self.split_name}_v3.npy')
        cnt_npy = os.path.join(cd, f'kps_count_{self.split_name}_v3.npy')

        if os.path.exists(kps_npy) and os.path.exists(cnt_npy):
            self.kps_data = np.load(kps_npy, mmap_mode='r')
            self.kps_count = np.load(cnt_npy, mmap_mode='r')
            print(f"[✓] Nạp có sẵn cache {self.split_name} ({self.length} mẫu .npy mmap 0-copy) < 0.05s.")
        else:
            print(f"[*] Preloading keypoints {self.split_name} ({self.length} mẫu) đa luồng an toàn...", flush=True)
            MAX_KPS_CACHE = 80
            kps_buf = np.zeros((self.length, 2, MAX_KPS_CACHE, 2), dtype=np.float32)
            cnt_buf = np.zeros(self.length, dtype=np.int32)

            def _load_single_kps(idx):
                pm = fm_list[idx]
                try:
                    kps = np.load(pm).astype(np.float32)
                    if kps.ndim == 3 and kps.shape[0] == 2 and kps.shape[2] == 2:
                        n = kps.shape[1]
                        if n > MAX_KPS_CACHE:
                            idx_sub = np.linspace(0, n - 1, MAX_KPS_CACHE, dtype=int)
                            kps_buf[idx] = kps[:, idx_sub, :]
                            cnt_buf[idx] = MAX_KPS_CACHE
                        elif n > 0:
                            kps_buf[idx, :, :n, :] = kps
                            cnt_buf[idx] = n
                except Exception:
                    pass

            with ThreadPoolExecutor(max_workers=4) as ex:
                list(tqdm(ex.map(_load_single_kps, range(self.length)), total=self.length, desc=f"   Loading KPS ({self.split_name})", leave=False))

            np.save(kps_npy, kps_buf)
            np.save(cnt_npy, cnt_buf)
            self.kps_data = np.load(kps_npy, mmap_mode='r')
            self.kps_count = np.load(cnt_npy, mmap_mode='r')
            del kps_buf, cnt_buf
            gc.collect()
            print(f"[✓] Đã tạo & mmap cache {self.split_name} ({self.length} mẫu) thành công.")

        # 3. Precompute TPS Cache cố định cho Validation (độ phân giải 540p -> 544 pad)
        if not self.is_train:
            val_c0_npy = os.path.join(cd, 'val_c0_v3.npy')
            val_th01_npy = os.path.join(cd, 'val_th01_v3.npy')
            val_c1_npy = os.path.join(cd, 'val_c1_v3.npy')
            val_th10_npy = os.path.join(cd, 'val_th10_v3.npy')
            val_valid_npy = os.path.join(cd, 'val_valid_v3.npy')

            all_val_exist = all(os.path.exists(p) for p in [val_c0_npy, val_th01_npy, val_c1_npy, val_th10_npy, val_valid_npy])
            if all_val_exist:
                self.val_c0 = np.load(val_c0_npy, mmap_mode='r')
                self.val_th01 = np.load(val_th01_npy, mmap_mode='r')
                self.val_c1 = np.load(val_c1_npy, mmap_mode='r')
                self.val_th10 = np.load(val_th10_npy, mmap_mode='r')
                self.val_valid = np.load(val_valid_npy, mmap_mode='r')
                print(f"[✓] Nạp có sẵn {self.length} mẫu TPS Validation mmap 0-copy (< 0.05s).")
            else:
                print(f"[*] Precomputing TPS cache cho Validation ({self.length} mẫu)...", flush=True)
                val_c0_buf = np.zeros((self.length, 88, 2), dtype=np.float32)
                val_th01_buf = np.zeros((self.length, 91, 2), dtype=np.float32)
                val_c1_buf = np.zeros((self.length, 88, 2), dtype=np.float32)
                val_th10_buf = np.zeros((self.length, 91, 2), dtype=np.float32)
                val_valid_buf = np.zeros((self.length, 1, 1, 1), dtype=np.float32)

                def _solve_single_val_tps(idx):
                    cnt = int(self.kps_count[idx])
                    kps = self.kps_data[idx, :, :cnt, :].copy() if cnt > 0 else np.zeros((2, 0, 2), dtype=np.float32)
                    w, h = 960, 540
                    pad_h = 544
                    pad_w = 960
                    if kps.shape[1] > 0:
                        kps_norm = kps.copy()
                        kps_norm[0, :, 0] = (kps_norm[0, :, 0] * w) / pad_w
                        kps_norm[0, :, 1] = (kps_norm[0, :, 1] * h) / pad_h
                        kps_norm[1, :, 0] = (kps_norm[1, :, 0] * w) / pad_w
                        kps_norm[1, :, 1] = (kps_norm[1, :, 1] * h) / pad_h
                    else:
                        kps_norm = kps
                    c0_p, th01_p, c1_p, th10_p, v = solve_tps_pair_cpu(kps_norm, max_kps=80, lambd=0.5, return_tensor=False)
                    val_c0_buf[idx] = c0_p
                    val_th01_buf[idx] = th01_p
                    val_c1_buf[idx] = c1_p
                    val_th10_buf[idx] = th10_p
                    val_valid_buf[idx, 0, 0, 0] = float(v)

                with ThreadPoolExecutor(max_workers=4) as ex:
                    list(tqdm(ex.map(_solve_single_val_tps, range(self.length)), total=self.length, desc="   Solving Val TPS", leave=False))

                np.save(val_c0_npy, val_c0_buf)
                np.save(val_th01_npy, val_th01_buf)
                np.save(val_c1_npy, val_c1_buf)
                np.save(val_th10_npy, val_th10_buf)
                np.save(val_valid_npy, val_valid_buf)
                self.val_c0 = np.load(val_c0_npy, mmap_mode='r')
                self.val_th01 = np.load(val_th01_npy, mmap_mode='r')
                self.val_c1 = np.load(val_c1_npy, mmap_mode='r')
                self.val_th10 = np.load(val_th10_npy, mmap_mode='r')
                self.val_valid = np.load(val_valid_npy, mmap_mode='r')
                del val_c0_buf, val_th01_buf, val_c1_buf, val_th10_buf, val_valid_buf
                gc.collect()
                print(f"[✓] Đã precompute và lưu {self.length} mẫu TPS Validation vào .npy mmap.")

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        clip_name = int(idx) if self.is_train else self.clips_arr[idx].decode('utf-8')
        im0 = imread_gray(self.p0_arr[idx].decode('utf-8'))
        imgt = imread_gray(self.pgt_arr[idx].decode('utf-8'))
        im1 = imread_gray(self.p1_arr[idx].decode('utf-8'))

        if im0 is None: im0 = np.full((540, 960), 255, dtype=np.uint8)
        if imgt is None: imgt = np.full((540, 960), 255, dtype=np.uint8)
        if im1 is None: im1 = np.full((540, 960), 255, dtype=np.uint8)

        h, w = im0.shape
        if imgt.shape != (h, w): imgt = cv2.resize(imgt, (w, h), interpolation=cv2.INTER_NEAREST)
        if im1.shape != (h, w): im1 = cv2.resize(im1, (w, h), interpolation=cv2.INTER_NEAREST)

        # Đệm biên lên bội số của 32 (540p -> 544x960) bảo toàn 100% chi tiết toàn khung hình
        pad_h = ((h + 31) // 32) * 32
        pad_w = ((w + 31) // 32) * 32
        if pad_h != h or pad_w != w:
            im0_pad = np.pad(im0, ((0, pad_h - h), (0, pad_w - w)), mode='constant', constant_values=255)
            imgt_pad = np.pad(imgt, ((0, pad_h - h), (0, pad_w - w)), mode='constant', constant_values=255)
            im1_pad = np.pad(im1, ((0, pad_h - h), (0, pad_w - w)), mode='constant', constant_values=255)
        else:
            im0_pad, imgt_pad, im1_pad = im0, imgt, im1

        if self.is_train:
            # Huấn luyện toàn phần 540p (Không crop)
            cnt = int(self.kps_count[idx])
            kps = self.kps_data[idx, :, :cnt, :].copy() if cnt > 0 else np.zeros((2, 0, 2), dtype=np.float32)

            if kps.shape[1] > 0:
                kps_norm = kps.copy()
                kps_norm[0, :, 0] = (kps_norm[0, :, 0] * w) / pad_w
                kps_norm[0, :, 1] = (kps_norm[0, :, 1] * h) / pad_h
                kps_norm[1, :, 0] = (kps_norm[1, :, 0] * w) / pad_w
                kps_norm[1, :, 1] = (kps_norm[1, :, 1] * h) / pad_h
            else:
                kps_norm = kps

            c0_pad, th01_pad, c1_pad, th10_pad, valid_flag = solve_tps_pair_cpu(kps_norm, max_kps=80, lambd=0.5, return_tensor=False)
            valid_flag = np.full((1, 1, 1), float(valid_flag), dtype=np.float32)
            t_scalar = float(self.t_arr[idx])
            if self.temporal_aug is not None:
                im0_pad, imgt_pad, im1_pad, t_scalar, kps_norm, c0_pad, th01_pad, c1_pad, th10_pad, _ = self.temporal_aug(
                    im0_pad, imgt_pad, im1_pad, t_scalar, kps_norm, c0_pad, th01_pad, c1_pad, th10_pad
                )

            return (
                torch.from_numpy(np.ascontiguousarray(im0_pad[None, :, :], dtype=np.uint8)),
                torch.from_numpy(np.ascontiguousarray(imgt_pad[None, :, :], dtype=np.uint8)),
                torch.from_numpy(np.ascontiguousarray(im1_pad[None, :, :], dtype=np.uint8)),
                torch.tensor(t_scalar, dtype=torch.float32),
                torch.from_numpy(c0_pad),
                torch.from_numpy(th01_pad),
                torch.from_numpy(c1_pad),
                torch.from_numpy(th10_pad),
                torch.from_numpy(valid_flag),
                clip_name,
                int(h),
                int(w)
            )
        else:
            # Thẩm định toàn phần 540p (Sử dụng precomputed cache)
            return (
                torch.from_numpy(np.ascontiguousarray(im0_pad[None, :, :], dtype=np.uint8)),
                torch.from_numpy(np.ascontiguousarray(imgt_pad[None, :, :], dtype=np.uint8)),
                torch.from_numpy(np.ascontiguousarray(im1_pad[None, :, :], dtype=np.uint8)),
                torch.tensor(float(self.t_arr[idx]), dtype=torch.float32),
                torch.from_numpy(self.val_c0[idx].copy()),
                torch.from_numpy(self.val_th01[idx].copy()),
                torch.from_numpy(self.val_c1[idx].copy()),
                torch.from_numpy(self.val_th10[idx].copy()),
                torch.from_numpy(self.val_valid[idx].copy()),
                clip_name,
                int(h),
                int(w)
            )

custom_collate_fn_v3 = default_collate

print("=" * 80)
print("[*] KHỞI TẠO DATASET VÀ DATALOADER 540P TOÀN PHẦN (FULL-FRAME 544x960, KHÔNG CROP, BATCH 8 + ACCUM 2)...")
train_dataset = SakugaGoldenDataset_v3(DATASET_ROOT, train_df, is_train=True, crop_size=None, cache_dir=CACHE_DIR)
val_dataset = SakugaGoldenDataset_v3(DATASET_ROOT, val_df, is_train=False, crop_size=None, cache_dir=CACHE_DIR)

train_weights = compute_v3_curriculum_weights(train_df)
train_sampler = WeightedRandomSampler(
    weights=torch.as_tensor(train_weights, dtype=torch.double),
    num_samples=len(train_dataset),
    replacement=True
)

BATCH_SIZE = 8   # 4 mẫu/GPU trên 2x T4 (VRAM an toàn < 9.2 GB ở 540p)
GRAD_ACCUM_STEPS = 2  # Tích lũy gradient 2 bước => Effective Batch Size = 16
VAL_BATCH_SIZE = 12  # 6 mẫu/GPU trên 2x T4 (Inference mode nhanh gấp đôi, VRAM < 3.2 GB)

def _worker_init_fn(worker_id):
    torch.set_num_threads(1)
    cv2.setNumThreads(0)
    random.seed(42 + worker_id)
    np.random.seed(42 + worker_id)

# [QUY TẮC SỐNG CÒN ZERO-LEAK]:
# 1. persistent_workers=False: Toàn bộ 4 worker bị huỷ và giải phóng 100% heap/RAM sau mỗi epoch!
# 2. pin_memory=False: Không dùng CUDA host caching allocator, triệt tiêu memory leak page-locked!
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=4,
    pin_memory=False,
    drop_last=True,
    persistent_workers=False,
    worker_init_fn=_worker_init_fn,
    collate_fn=custom_collate_fn_v3
)

val_loader = DataLoader(
    val_dataset,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=False,
    drop_last=False,
    persistent_workers=False,
    worker_init_fn=_worker_init_fn,
    collate_fn=custom_collate_fn_v3
)

print(f"[✓] DataLoaders sẵn sàng: Train={len(train_loader)} batches (Batch={BATCH_SIZE}, Accum={GRAD_ACCUM_STEPS}) | Val={len(val_loader)} batches")
print("=" * 80)
sys.stdout.flush()


In [ ]:
# [CELL 5]: Kiến Trúc Mạng TPS-Inbetween V3.0 (Asymmetric Stroke & Occlusion Engine)
import os
import sys
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

def conv(in_p, out_p, kernel_size=3, stride=1, padding=1, dilation=1):
    return nn.Sequential(
        nn.Conv2d(in_p, out_p, kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation, bias=True),
        nn.PReLU(out_p)
    )

def deconv(in_p, out_p, kernel_size=4, stride=2, padding=1):
    return nn.Sequential(
        nn.ConvTranspose2d(in_p, out_p, kernel_size=kernel_size, stride=stride, padding=padding, bias=True),
        nn.PReLU(out_p)
    )

class Conv2(nn.Module):
    def __init__(self, in_p, out_p, stride=2):
        super(Conv2, self).__init__()
        self.conv1 = conv(in_p, out_p, 3, stride, 1)
        self.conv2 = conv(out_p, out_p, 3, 1, 1)

    def forward(self, x):
        return self.conv2(self.conv1(x))

class Contextnet(nn.Module):
    def __init__(self, indim=1, c=16):
        super(Contextnet, self).__init__()
        self.conv1 = Conv2(indim, c)
        self.conv2 = Conv2(c, 2*c)
        self.conv3 = Conv2(2*c, 4*c)
        self.conv4 = Conv2(4*c, 8*c)

    def forward(self, x, flow):
        x = self.conv1(x)
        flow = F.interpolate(flow, scale_factor=0.5, mode='bilinear', align_corners=False) * 0.5
        f1 = warp_normalized_flow(x, flow)
        x = self.conv2(x)
        flow = F.interpolate(flow, scale_factor=0.5, mode='bilinear', align_corners=False) * 0.5
        f2 = warp_normalized_flow(x, flow)
        x = self.conv3(x)
        flow = F.interpolate(flow, scale_factor=0.5, mode='bilinear', align_corners=False) * 0.5
        f3 = warp_normalized_flow(x, flow)
        x = self.conv4(x)
        flow = F.interpolate(flow, scale_factor=0.5, mode='bilinear', align_corners=False) * 0.5
        f4 = warp_normalized_flow(x, flow)
        return [f1, f2, f3, f4]

class IFBlock(nn.Module):
    def __init__(self, in_planes, c=64):
        super(IFBlock, self).__init__()
        self.conv0 = nn.Sequential(
            conv(in_planes, c//2, 3, 2, 1),
            conv(c//2, c, 3, 2, 1),
        )
        self.convblock = nn.Sequential(*[conv(c, c) for _ in range(8)])
        self.lastconv = nn.ConvTranspose2d(c, 5, 4, 2, 1)

    def forward(self, x, flow, scale):
        if scale != 1:
            x = F.interpolate(x, scale_factor=1./scale, mode='bilinear', align_corners=False)
        if flow is not None:
            flow = F.interpolate(flow, scale_factor=1./scale, mode='bilinear', align_corners=False) * (1./scale)
            x = torch.cat((x, flow), 1)
        x = self.conv0(x.contiguous())
        x = self.convblock(x) + x
        with torch.amp.autocast('cuda', enabled=False):
            tmp = self.lastconv(x.float().contiguous())
            tmp = F.interpolate(tmp, scale_factor=scale * 2, mode='bilinear', align_corners=False)
            flow = tmp[:, :4] * scale * 2
            mask = tmp[:, 4:5]
        return flow, mask

class IFNet(nn.Module):
    def __init__(self):
        super(IFNet, self).__init__()
        self.block0 = IFBlock(6, c=240)
        self.block1 = IFBlock(13+4, c=150)
        self.block2 = IFBlock(13+4, c=90)

    def forward(self, I0t, I1t, scale=[4, 2, 1]):
        x = torch.cat((I0t.repeat(1, 3, 1, 1), I1t.repeat(1, 3, 1, 1)), dim=1)
        img0 = x[:, :3]
        img1 = x[:, 3:6]
        flow_list = []
        mask_list = []
        warped_img0 = img0
        warped_img1 = img1
        flow = None
        stu = [self.block0, self.block1, self.block2]
        for i in range(3):
            if flow is not None:
                flow_d, mask_d = stu[i](torch.cat((img0, img1, warped_img0, warped_img1, mask), 1), flow, scale=scale[i])
                flow = flow + flow_d
                mask = mask + mask_d
            else:
                flow, mask = stu[i](torch.cat((img0, img1), 1), None, scale=scale[i])
            mask_list.append(torch.sigmoid(mask))
            flow_list.append(flow)
            warped_img0 = warp_pixel_flow(img0, flow[:, :2])
            warped_img1 = warp_pixel_flow(img1, flow[:, 2:4])
        return flow_list[2], mask_list[2]

# =============================================================================
# [CƠ CHẾ CỐT LÕI 1 - V3.0]: Bản Đồ Bất Đối Xứng Đặc Trưng & Occlusion Confidence
# =============================================================================
class AsymmetricFeatureAnalyzer(nn.Module):
    def __init__(self, in_channels=34):  # 1 (I0) + 1 (I1) + 16 (f0[0]) + 16 (f1[0])
        super(AsymmetricFeatureAnalyzer, self).__init__()
        self.net = nn.Sequential(
            conv(in_channels, 32, 3, 1, 1),
            conv(32, 16, 3, 1, 1),
            nn.Conv2d(16, 2, 3, 1, 1)
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.constant_(self.net[-1].bias, -1.5)

    def forward(self, fine_I0t, fine_I1t, feat0_sub, feat1_sub):
        b, _, h, w = fine_I0t.shape
        f0_up = F.interpolate(feat0_sub, size=(h, w), mode='bilinear', align_corners=False)
        f1_up = F.interpolate(feat1_sub, size=(h, w), mode='bilinear', align_corners=False)
        inp = torch.cat([fine_I0t, fine_I1t, f0_up, f1_up], dim=1)
        occ_logits = self.net(inp)
        occ_confidence = torch.sigmoid(occ_logits)
        return occ_confidence

# =============================================================================
# [CƠ CHẾ CỐT LÕI 2 - V3.0]: Lan Truyền Vận Tốc Theo Đường Biên Bao (Contour-Guided Flow)
# =============================================================================
class ContourGuidedFlowPropagator(nn.Module):
    def __init__(self):
        super(ContourGuidedFlowPropagator, self).__init__()
        self.conv_in = conv(7, 32, 3, 1, 1)  # 1 (I0) + 1 (I1) + 4 (flow) + 1 (shared contour)
        self.d1 = conv(32, 32, 3, 1, 1, dilation=1)
        self.d2 = conv(32, 32, 3, 1, 2, dilation=2)
        self.d4 = conv(32, 32, 3, 1, 4, dilation=4)
        self.conv_out = nn.Conv2d(32, 4, 3, 1, 1)
        nn.init.zeros_(self.conv_out.weight)
        nn.init.zeros_(self.conv_out.bias)

    def forward(self, coarse_I0t, coarse_I1t, flow_refine):
        shared_contour = torch.minimum(coarse_I0t, coarse_I1t)
        inp = torch.cat([coarse_I0t, coarse_I1t, flow_refine, shared_contour], dim=1)
        x = self.conv_in(inp)
        x = self.d1(x) + self.d2(x) + self.d4(x)
        delta_flow = self.conv_out(x)
        return flow_refine + delta_flow, delta_flow

# =============================================================================
# [CƠ CHẾ CỐT LÕI 3 - V3.0]: Cổng Hiển Thị Kép Độc Lập (Phá Bỏ Bẫy Tổng Bằng 1)
# =============================================================================
class AsymmetricStrokeFusionHead_v3(nn.Module):
    def __init__(self):
        super(AsymmetricStrokeFusionHead_v3, self).__init__()
        self.net = nn.Sequential(
            conv(10, 32, 3, 1, 1),
            conv(32, 32, 3, 1, 1),
            nn.Conv2d(32, 2, 3, 1, 1)
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)
        self.gamma_occ = nn.Parameter(torch.full((1, 2, 1, 1), 0.35))

    def forward(self, fine_I0t, fine_I1t, flow, ifnet_mask, t_vals, occ_confidence):
        b, _, h, w = fine_I0t.shape
        t_map = t_vals.repeat(1, 1, h, w)
        inp = torch.cat([fine_I0t, fine_I1t, flow, ifnet_mask, t_map, occ_confidence], dim=1)
        residual_logits = self.net(inp)

        prior_0 = torch.logit(torch.clamp(1.0 - t_map, 1e-4, 1.0 - 1e-4))
        prior_1 = torch.logit(torch.clamp(t_map, 1e-4, 1.0 - 1e-4))
        prior_logits = torch.cat([prior_0, prior_1], dim=1)

        vis_logits = prior_logits + residual_logits + self.gamma_occ * occ_confidence
        visibility_gates = torch.sigmoid(vis_logits)
        v0 = visibility_gates[:, 0:1]
        v1 = visibility_gates[:, 1:2]
        return v0, v1

class SynthesisUnet_v3(nn.Module):
    def __init__(self, indim=9, c=16):
        super(SynthesisUnet_v3, self).__init__()
        self.down0 = Conv2(indim, 2*c)
        self.down1 = Conv2(4*c, 4*c)
        self.down2 = Conv2(8*c, 8*c)
        self.down3 = Conv2(16*c, 16*c)
        self.up0 = deconv(32*c, 8*c)
        self.up1 = deconv(16*c, 4*c)
        self.up2 = deconv(8*c, 2*c)
        self.up3 = deconv(4*c, c)

        self.conv_erase = nn.Conv2d(c, 1, 3, 1, 1)
        self.conv_add = nn.Conv2d(c, 1, 3, 1, 1)

    def forward(self, img0, img1, flow0, flow1, warped0, warped1, mask, c0, c1):
        s0 = self.down0(torch.cat((img0, img1, warped0, warped1, flow0, flow1, mask), 1))
        s1 = self.down1(torch.cat((s0, c0[0], c1[0]), 1))
        s2 = self.down2(torch.cat((s1, c0[1], c1[1]), 1))
        s3 = self.down3(torch.cat((s2, c0[2], c1[2]), 1))
        x = self.up0(torch.cat((s3, c0[3], c1[3]), 1))
        x = self.up1(torch.cat((x, s2), 1))
        x = self.up2(torch.cat((x, s1), 1))
        x = self.up3(torch.cat((x, s0), 1))

        gate_erase = torch.sigmoid(self.conv_erase(x))
        detail_add = torch.sigmoid(self.conv_add(x))
        return gate_erase, detail_add

class TPSInbetweenFinetune_v3(nn.Module):
    """
    Kiến trúc TPS-Inbetween V3.0 (Major Architectural Upgrade).
    Tương thích 100% Warm-Start Backbone V2.6 + 3 Nhánh Đột Phá V3:
    1. AsymmetricFeatureAnalyzer (Occlusion Confidence)
    2. ContourGuidedFlowPropagator (Biên bao dẫn hướng)
    3. AsymmetricStrokeFusionHead_v3 (Decoupled Visibility Gates V0, V1)
    4. SynthesisUnet_v3 (Gated Silhouette Erase & Hair-tip Reconnection)
    """
    def __init__(self):
        super(TPSInbetweenFinetune_v3, self).__init__()
        self.encoder = Contextnet(indim=1)
        self.IFNet = IFNet()
        self.asym_analyzer = AsymmetricFeatureAnalyzer(in_channels=34)
        self.contour_propagator = ContourGuidedFlowPropagator()
        self.asf_head = AsymmetricStrokeFusionHead_v3()
        self.unet = SynthesisUnet_v3(indim=9)

        self._init_v3_weights()

    def _init_v3_weights(self):
        for m in [self.asym_analyzer, self.contour_propagator, self.asf_head]:
            for mod in m.modules():
                if isinstance(mod, (nn.Conv2d, nn.ConvTranspose2d)):
                    nn.init.kaiming_normal_(mod.weight, a=0.25, mode='fan_out', nonlinearity='leaky_relu')
                    if mod.bias is not None:
                        nn.init.zeros_(mod.bias)
        # Small normal trên các lớp output để phá bẫy đối xứng mà vẫn giữ ổn định
        for head in [self.asf_head.net[-1], self.contour_propagator.conv_out]:
            if hasattr(head, 'weight') and head.weight is not None:
                nn.init.normal_(head.weight, mean=0.0, std=1e-3)
                if head.bias is not None:
                    nn.init.zeros_(head.bias)

    def forward(self, x0, x1, t_vals, kps_input=None, kps_counts=None, max_kps=80, tps_precomputed=None):
        if x0.dtype == torch.uint8:
            x0 = (255.0 - x0.float()) * (1.0 / 255.0)
        if x1.dtype == torch.uint8:
            x1 = (255.0 - x1.float()) * (1.0 / 255.0)

        b, c, h, w = x0.shape
        dev = x0.device
        t_vals = t_vals.view(b, 1, 1, 1).to(dev)

        # 1. Coarse TPS Motion
        if tps_precomputed is not None:
            c0_b, th01_b, c1_b, th10_b, valid_b = tps_precomputed
            c0_b = c0_b.to(dev)
            th01_b = th01_b.to(dev)
            c1_b = c1_b.to(dev)
            th10_b = th10_b.to(dev)
            valid_b = valid_b.to(dev)

            coarse_flow01 = tps_flow_from_theta_batch(th01_b, c0_b, valid_b, h, w, dev)
            coarse_flow10 = tps_flow_from_theta_batch(th10_b, c1_b, valid_b, h, w, dev)
        else:
            flow01_list, flow10_list = [], []
            with torch.no_grad():
                for i in range(b):
                    kps = kps_input[i] if kps_input is not None else None
                    c0_p, th01_p, c1_p, th10_p, v = solve_tps_pair_cpu(kps, max_kps=max_kps, lambd=0.5)
                    c0_p = c0_p.unsqueeze(0).to(dev)
                    th01_p = th01_p.unsqueeze(0).to(dev)
                    c1_p = c1_p.unsqueeze(0).to(dev)
                    th10_p = th10_p.unsqueeze(0).to(dev)
                    v_t = v.view(1, 1, 1, 1).to(dev)

                    f01 = tps_flow_from_theta_batch(th01_p, c0_p, v_t, h, w, dev)
                    f10 = tps_flow_from_theta_batch(th10_p, c1_p, v_t, h, w, dev)
                    flow01_list.append(f01)
                    flow10_list.append(f10)

            coarse_flow01 = torch.cat(flow01_list, dim=0)
            coarse_flow10 = torch.cat(flow10_list, dim=0)

        coarse_flow0t = coarse_flow01 * (t_vals ** 2) - (1.0 - t_vals) * t_vals * coarse_flow10
        coarse_flow1t = coarse_flow10 * ((1.0 - t_vals) ** 2) - t_vals * (1.0 - t_vals) * coarse_flow01

        # 2. Coarse Warping
        coarse_I0t = warp_normalized_flow(x0, coarse_flow0t)
        coarse_I1t = warp_normalized_flow(x1, coarse_flow1t)

        # 3. Fine Motion & Feature Extraction
        feats0 = self.encoder(x0, coarse_flow0t)
        feats1 = self.encoder(x1, coarse_flow1t)
        flow_raw, ifnet_mask = self.IFNet(coarse_I0t, coarse_I1t)

        # [V3.0 UPGRADE 1]: Lan truyền vận tốc theo đường biên bao (Contour-Guided Flow)
        flow_guided, delta_flow = self.contour_propagator(coarse_I0t, coarse_I1t, flow_raw)

        # 4. Fine Warping với Flow được dẫn hướng
        fine_I0t = warp_pixel_flow(coarse_I0t, flow_guided[:, :2])
        fine_I1t = warp_pixel_flow(coarse_I1t, flow_guided[:, 2:4])

        # [V3.0 UPGRADE 2]: Phân tích bất đối xứng đặc trưng & Occlusion Confidence
        occ_conf = self.asym_analyzer(fine_I0t, fine_I1t, feats0[0], feats1[0])

        # [V3.0 UPGRADE 3]: Cổng hiển thị kép độc lập (V0, V1) in [0, 1]^2
        v0, v1 = self.asf_head(fine_I0t, fine_I1t, flow_guided, ifnet_mask, t_vals, occ_conf)
        base_interp = torch.clamp(v0 * fine_I0t + v1 * fine_I1t, 0.0, 1.0)

        # 5. Gated Silhouette Erase & Detail Reconnection (UNet v3.0)
        gate_erase, detail_add = self.unet(
            x0, x1,
            coarse_flow0t, coarse_flow1t,
            fine_I0t, fine_I1t, ifnet_mask, feats0, feats1
        )
        pred_t = torch.clamp(gate_erase * base_interp + detail_add, 0.0, 1.0)
        if self.training:
            return pred_t, (v0, v1), occ_conf, delta_flow, fine_I0t, fine_I1t
        return pred_t, (v0, v1), occ_conf

# Alias tương thích
base_model = TPSInbetweenFinetune_v3().to(device)
print("=" * 80)
print("[✓] Mô hình TPSInbetweenFinetune_v3 (Asymmetric Stroke & Occlusion Engine) sẵn sàng.")
print("=" * 80)
sys.stdout.flush()


In [ ]:
# [CELL 6]: Bộ Hàm Mất Mát CompositeLossEngine_v30 (2-Stage Curriculum, Gold 1.10x & Hair-Tip Loss)
import os
import sys
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
def soft_erode(img):
    p1 = -F.max_pool2d(-img, (3, 1), (1, 1), (1, 0))
    p2 = -F.max_pool2d(-img, (1, 3), (1, 1), (0, 1))
    return torch.min(p1, p2)

def soft_dilate(img):
    return F.max_pool2d(img, (3, 3), (1, 1), (1, 1))

def soft_open(img):
    return soft_dilate(soft_erode(img))

def soft_skeletonize(img, iters=3):
    img = torch.clamp(img, 0.0, 1.0)
    skel = F.relu(img - soft_open(img))
    for _ in range(iters - 1):
        img = soft_erode(img)
        skel = skel + F.relu(img - soft_open(img))
    return torch.clamp(skel, 0.0, 1.0)

class SoftClDiceLoss(nn.Module):
    def __init__(self, iters=3, smooth=1.0):
        super(SoftClDiceLoss, self).__init__()
        self.iters = iters
        self.smooth = smooth

    def forward(self, pred, target):
        p_skel = soft_skeletonize(pred, iters=self.iters)
        t_skel = soft_skeletonize(target, iters=self.iters)
        tprec = (torch.sum(p_skel * target) + self.smooth) / (torch.sum(p_skel) + self.smooth)
        tsens = (torch.sum(t_skel * pred) + self.smooth) / (torch.sum(t_skel) + self.smooth)
        cldice = 2.0 * (tprec * tsens) / (tprec + tsens + 1e-6)
        return 1.0 - cldice

class StrokeDensityRegularizationLoss_v3(nn.Module):
    def __init__(self, target_min=1.04, target_max=1.14, target_center=1.10, ramp_weight=0.25):
        super().__init__()
        self.target_min = target_min
        self.target_max = target_max
        self.target_center = target_center
        self.ramp_weight = ramp_weight

    def forward(self, pred_ink, gt_ink):
        p_ink = torch.clamp(pred_ink, 0.0, 1.0)
        g_ink = torch.clamp(gt_ink, 0.0, 1.0)
        m_soft_p = torch.clamp((p_ink - 0.06) / 0.18, 0.0, 1.0) + self.ramp_weight * F.relu(p_ink - 0.24)
        m_soft_g = torch.clamp((g_ink - 0.06) / 0.18, 0.0, 1.0) + self.ramp_weight * F.relu(g_ink - 0.24)
        m_hard_p = (p_ink > 0.15).float()
        m_hard_g = (g_ink > 0.15).float()
        m_ste_p = m_hard_p + (m_soft_p - m_soft_p.detach())
        m_ste_g = m_hard_g + (m_soft_g - m_soft_g.detach())
        sum_p = torch.sum(m_ste_p, dim=[-3, -2, -1])
        sum_g = torch.sum(m_ste_g, dim=[-3, -2, -1])
        ratio = sum_p / torch.clamp(sum_g, min=10.0)
        dev_low = F.relu(self.target_min - ratio)
        dev_high = F.relu(ratio - self.target_max)
        loss_low = dev_low ** 2
        loss_high = 2.5 * dev_high + 1.5 * (dev_high ** 2)
        center_pull = 0.05 * ((ratio - self.target_center) ** 2)
        return torch.mean(loss_low + loss_high + center_pull)

class HairTipEndpointContinuityLoss(nn.Module):
    def __init__(self, endpoint_focal=4.0):
        super().__init__()
        self.endpoint_focal = endpoint_focal

    def forward(self, pred_ink, gt_ink):
        p_ink = torch.clamp(pred_ink, 0.0, 1.0)
        g_ink = torch.clamp(gt_ink, 0.0, 1.0)
        skel_gt = soft_skeletonize(g_ink, iters=3)
        nbrs = F.avg_pool2d(skel_gt, kernel_size=3, stride=1, padding=1) * 9.0 - skel_gt
        endpoints = (nbrs <= 1.8) & (skel_gt > 0.20)
        hair_tip_roi = soft_dilate(endpoints.float())
        tip_under = F.relu(g_ink - p_ink) * hair_tip_roi
        return self.endpoint_focal * torch.mean(tip_under)

class OcclusionAwareGhostLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, pred, gt, v0, v1):
        pred_f = torch.clamp(pred, 0.0, 1.0)
        gt_f = torch.clamp(gt, 0.0, 1.0)
        outside_gt = (gt_f < 0.05).float()
        far_ghost = F.relu(pred_f - 0.03) * outside_gt
        dark_ghost = F.relu(pred_f - 0.30) * outside_gt
        cleanliness_penalty = 0.35 * F.relu(pred_f - 0.008) * outside_gt
        gate_penalty = F.relu(v0 + v1 - 1.05)
        return 1.2 * torch.mean(far_ghost) + 3.0 * torch.mean(dark_ghost) + torch.mean(cleanliness_penalty) + 0.10 * torch.mean(gate_penalty)

class CompositeLossEngine_v30(nn.Module):
    def __init__(self):
        super(CompositeLossEngine_v30, self).__init__()
        self.cldice = SoftClDiceLoss(iters=3)
        self.density_loss = StrokeDensityRegularizationLoss_v3(target_min=1.04, target_max=1.14, target_center=1.10)
        self.hair_tip_loss = HairTipEndpointContinuityLoss()
        self.ghost_loss = OcclusionAwareGhostLoss()

    def forward(self, pred, target, v0=None, v1=None, epoch=1):
        loss_l1 = F.l1_loss(pred, target)
        loss_dens = self.density_loss(pred, target)
        loss_hair = self.hair_tip_loss(pred, target)

        if epoch <= 2:
            # Giai đoạn 1 Warmup: Tập trung căn chỉnh L1, Density và HairTip
            total = 1.0 * loss_l1 + 0.55 * loss_dens + 0.30 * loss_hair
            loss_dice = torch.tensor(0.0, device=pred.device)
        else:
            # Giai đoạn 2: Bật toàn bộ clDice và Ghost Suppression
            loss_dice = self.cldice(pred, target)
            loss_ghost = self.ghost_loss(pred, target, v0, v1) if (v0 is not None and v1 is not None) else torch.tensor(0.0, device=pred.device)
            total = 1.0 * loss_l1 + 0.15 * loss_dice + 0.55 * loss_dens + 0.30 * loss_hair + 0.20 * loss_ghost

        return total, loss_l1, loss_dice, loss_dens, loss_hair

criterion = CompositeLossEngine_v30().to(device)
print("[✓] CompositeLossEngine_v30 sẵn sàng (2-Stage Curriculum + Gold 1.10x + Hair-Tip Loss + Cleanliness Penalty).")
sys.stdout.flush()

In [ ]:
# [CELL 7]: Nạp Checkpoint Warm-Start V2.6 (Bộ Lọc An Toàn asf_head) & Parallel Training Wrapper
import os
import sys
import glob
import time
import torch
import torch.nn as nn

print("=" * 80)
print("[*] ĐANG TÌM KIẾM VÀ NẠP CHECKPOINT WARM-START V2.6...")
t_ckpt_start = time.time()

if 'KAGGLE_INPUT' not in globals():
    KAGGLE_INPUT = '/kaggle/input' if os.path.exists('/kaggle/input') else './data'

# In ra chi tiết cây thư mục input để kiểm tra trực quan
if os.path.exists(KAGGLE_INPUT):
    try:
        top_entries = os.listdir(KAGGLE_INPUT)
        print(f"[*] Các mục tại {KAGGLE_INPUT}: {top_entries}")
        if 'datasets' in top_entries:
            ds_path = os.path.join(KAGGLE_INPUT, 'datasets')
            try:
                ds_entries = os.listdir(ds_path)
                print(f"[*] Các mục trong {ds_path}: {ds_entries}")
                for entry in ds_entries[:5]:
                    sub_p = os.path.join(ds_path, entry)
                    if os.path.isdir(sub_p):
                        print(f"    - {entry}: {os.listdir(sub_p)}")
            except Exception as e:
                print(f"[*] Không thể listdir {ds_path}: {e}")
    except Exception as e:
        print(f"[*] Không thể listdir {KAGGLE_INPUT}: {e}")

# 1. Tìm kiếm nhanh bằng Glob mẫu đường dẫn Kaggle Datasets (cả layout cũ và mới)
discovered_pts = []
direct_patterns = [
    os.path.join(KAGGLE_INPUT, "datasets", "*", "*", "model_tps_v2.6_best.pt"),
    os.path.join(KAGGLE_INPUT, "datasets", "*", "model_tps_v2.6_best.pt"),
    os.path.join(KAGGLE_INPUT, "*", "model_tps_v2.6_best.pt"),
    os.path.join(KAGGLE_INPUT, "datasets", "*", "*", "*.pt"),
    os.path.join(KAGGLE_INPUT, "datasets", "*", "*.pt"),
    os.path.join(KAGGLE_INPUT, "*", "*.pt"),
    os.path.join(WORKING_DIR, "checkpoints_v3.0", "*.pt"),
    "./checkpoints/experiments/v2.6/checkpoints_v2.6/model_tps_v2.6_best.pt"
]

for pat in direct_patterns:
    for f in glob.glob(pat):
        f_norm = os.path.normpath(f)
        if f_norm not in discovered_pts:
            discovered_pts.append(f_norm)

# 2. Quét sâu có tỉa nhánh (Pruned Walk) - TUYỆT ĐỐI KHÔNG tỉa 'datasets' hay 'checkpoints'
def should_skip_dir(d_name):
    d_low = d_name.lower()
    # Tỉa dataset ảnh sakuga để tránh duyệt 35.000 file
    if 'sakuga' in d_low:
        return True
    # Tỉa các thư mục chứa khung hình, clip, cache
    if d_low in {'frames', 'clips', 'matches', 'crops', 'images', 'npy', 'cache', '.git', '__pycache__'}:
        return True
    return False

search_roots = [KAGGLE_INPUT, WORKING_DIR, './checkpoints', '.']
for s_root in search_roots:
    if not os.path.exists(s_root):
        continue
    for root, dirs, files in os.walk(s_root):
        dirs[:] = [d for d in dirs if not should_skip_dir(d)]
        for f in files:
            if f.endswith('.pt') or f.endswith('.pth'):
                p_norm = os.path.normpath(os.path.join(root, f))
                if p_norm not in discovered_pts:
                    discovered_pts.append(p_norm)

# 3. Chấm điểm ưu tiên tự động cho các checkpoint tìm thấy
def score_ckpt(path):
    p = path.lower().replace('\\', '/')
    name = os.path.basename(p)
    score = 0
    if 'model_tps_v2.6_best.pt' in name:
        score += 1000
    elif 'v2.6' in p and 'best' in name:
        score += 800
    elif 'v2.6' in p:
        score += 600
    elif 'v2.5' in p and 'best' in name:
        score += 400
    elif 'v2.5' in p:
        score += 300
    elif 'best' in name:
        score += 200
    elif name.endswith('.pt'):
        score += 100
    return score

discovered_pts.sort(key=score_ckpt, reverse=True)

clean_sd = None
loaded_path = None

for cand in discovered_pts:
    if os.path.exists(cand) and os.path.isfile(cand):
        try:
            print(f"    [*] Đang đọc thử checkpoint: {cand} (Điểm ưu tiên: {score_ckpt(cand)})...")
            raw_sd = torch.load(cand, map_location="cpu")
            if isinstance(raw_sd, dict) and "model_state_dict" in raw_sd:
                raw_sd = raw_sd["model_state_dict"]
            elif isinstance(raw_sd, dict) and "model" in raw_sd:
                raw_sd = raw_sd["model"]
            clean_sd = {k.replace("module.", "").replace("base_model.", ""): v for k, v in raw_sd.items()}
            loaded_path = cand
            print(f"    [✓] Đọc thành công checkpoint ({len(clean_sd)} tensors) từ: {cand}")
            break
        except Exception as e:
            print(f"    [!] Bỏ qua {cand}: {e}")

model_sd = base_model.state_dict()
matched_sd = {}
if clean_sd is not None:
    for k, v in clean_sd.items():
        if k in model_sd and model_sd[k].shape == v.shape:
            # [QUAN TRỌNG]: Bỏ qua toàn bộ asf_head.* cũ từ V2.x để nhánh V3 mới giữ nguyên Kaiming Init
            if k.startswith("asf_head."):
                continue
            matched_sd[k] = v
    base_model.load_state_dict(matched_sd, strict=False)
    print(f"[✓] Nạp hoàn tất {len(matched_sd)}/{len(model_sd)} tensors từ Warm-Start V2.6 ({time.time() - t_ckpt_start:.2f}s).")
    print(f"    - Nguồn checkpoint: {loaded_path}")
    print("    - Nhánh asf_head, contour_propagator và asym_analyzer V3 được bảo toàn khởi tạo Kaiming Normal (a=0.25).")
else:
    print(f"⚠️ [CẢNH BÁO]: Không tìm thấy checkpoint cũ trong {time.time() - t_ckpt_start:.2f}s!")
    print(f"    - Thư mục tìm kiếm: {KAGGLE_INPUT}")
    if os.path.exists(KAGGLE_INPUT):
        print(f"    - Các mục hiện diện trong {KAGGLE_INPUT}: {os.listdir(KAGGLE_INPUT)}")
    print("    - Tạm thời khởi chạy từ trọng số mặc định (Random Weights).")

class ParallelTrainingWrapper_v3(nn.Module):
    def __init__(self, model, criterion, use_amp=True):
        super(ParallelTrainingWrapper_v3, self).__init__()
        self.model = model
        self.criterion = criterion
        self.use_amp = use_amp
        self.current_epoch = 1

    def forward(self, img0, imggt, img1, t_val, c0, th01, c1, th10, valid, compute_loss=True):
        dev = img0.device
        with torch.amp.autocast('cuda', enabled=(self.use_amp and compute_loss)):
            res = self.model(img0, img1, t_val, tps_precomputed=(c0, th01, c1, th10, valid))
            pred = res[0]
            v0, v1 = res[1]
            if compute_loss:
                if imggt.dtype == torch.uint8:
                    imggt = ((255.0 - imggt.float()) * (1.0 / 255.0)).contiguous()
                total, l1, dice, dens, hair = self.criterion(pred, imggt, v0, v1, epoch=self.current_epoch)
                return total.unsqueeze(0), l1.unsqueeze(0), dice.unsqueeze(0), dens.unsqueeze(0), hair.unsqueeze(0)
            return pred

raw_wrapper = ParallelTrainingWrapper_v3(base_model, criterion, use_amp=True)
if NUM_GPUS > 1:
    parallel_engine = nn.DataParallel(raw_wrapper, device_ids=list(range(NUM_GPUS)))
    print(f"[✓] Đã bọc mô hình trong nn.DataParallel ({NUM_GPUS} GPUs T4 sẵn sàng).")
else:
    parallel_engine = raw_wrapper
    print("[✓] Chạy trên thiết bị đơn GPU.")

print("=" * 80)
sys.stdout.flush()


In [ ]:
# [CELL 8]: Hàm Đo Lường Chỉ Số Thẩm Định Chống Bẫy V3.0 (100% Batched Pure GPU Metrics)
import os
import sys
import math
import torch
import torch.nn.functional as F
def _create_gaussian_1d_kernel(kernel_size=11, sigma=1.5, device='cuda'):
    coords = torch.arange(kernel_size, dtype=torch.float32, device=device) - (kernel_size - 1) / 2.0
    g = torch.exp(-(coords ** 2) / (2.0 * sigma ** 2))
    g = g / g.sum()
    g_x = g.view(1, 1, 1, kernel_size).repeat(5, 1, 1, 1)
    g_y = g.view(1, 1, kernel_size, 1).repeat(5, 1, 1, 1)
    return g_x, g_y

_GAUSS_X = None
_GAUSS_Y = None

def calculate_metrics_batch_gpu(pred_tensor, gt_tensor):
    global _GAUSS_X, _GAUSS_Y
    with torch.no_grad():
        p = torch.clamp(pred_tensor.float(), 0.0, 1.0)
        g = torch.clamp(gt_tensor.float(), 0.0, 1.0)
        b, c, h, w = p.shape
        dev = p.device

        # 1. Batched PSNR
        mse = torch.mean((p - g) ** 2, dim=[-3, -2, -1])
        psnr = 10.0 * torch.log10(1.0 / torch.clamp(mse, min=1e-8))

        # 2. Batched GPU SSIM
        if _GAUSS_X is None or _GAUSS_X.device != dev:
            _GAUSS_X, _GAUSS_Y = _create_gaussian_1d_kernel(11, 1.5, dev)
        
        gx = _GAUSS_X[:c] if _GAUSS_X.shape[0] >= c else _GAUSS_X[0:1].repeat(c, 1, 1, 1)
        gy = _GAUSS_Y[:c] if _GAUSS_Y.shape[0] >= c else _GAUSS_Y[0:1].repeat(c, 1, 1, 1)

        def _conv2d_separable(t):
            t_pad = F.pad(t, (5, 5, 5, 5), mode='reflect')
            return F.conv2d(F.conv2d(t_pad, gx, groups=c), gy, groups=c)

        mu_p = _conv2d_separable(p)
        mu_g = _conv2d_separable(g)
        mu_p_sq = mu_p ** 2
        mu_g_sq = mu_g ** 2
        mu_pg = mu_p * mu_g
        sigma_p_sq = _conv2d_separable(p * p) - mu_p_sq
        sigma_g_sq = _conv2d_separable(g * g) - mu_g_sq
        sigma_pg = _conv2d_separable(p * g) - mu_pg
        c1_s = 0.01 ** 2
        c2_s = 0.03 ** 2
        ssim_map = ((2.0 * mu_pg + c1_s) * (2.0 * sigma_pg + c2_s)) / ((mu_p_sq + mu_g_sq + c1_s) * (sigma_p_sq + sigma_g_sq + c2_s))
        ssim = torch.mean(ssim_map, dim=[-3, -2, -1])

        # 3. Stroke Recall & F1
        p_bin = (p > 0.15).float()
        g_bin = (g > 0.15).float()
        tp = torch.sum(p_bin * g_bin, dim=[-3, -2, -1])
        fn = torch.sum((1.0 - p_bin) * g_bin, dim=[-3, -2, -1])
        fp = torch.sum(p_bin * (1.0 - g_bin), dim=[-3, -2, -1])
        recall = tp / torch.clamp(tp + fn, min=1e-5)
        f1 = (2.0 * tp) / torch.clamp(2.0 * tp + fp + fn, min=1e-5)

        # 4. Density Ratio
        sum_p = torch.sum(p_bin, dim=[-3, -2, -1])
        sum_g = torch.sum(g_bin, dim=[-3, -2, -1])
        density = sum_p / torch.clamp(sum_g, min=1.0)

        # 5. Ghost Pixels & Dark Ghost
        g_dilated = soft_dilate(g_bin)
        outside_core = 1.0 - g_dilated
        ghost_px = torch.sum((p > 0.10).float() * outside_core, dim=[-3, -2, -1])
        dark_ghost_px = torch.sum((p > 0.30).float() * outside_core, dim=[-3, -2, -1])

        # 6. Approximated Chamfer
        p_sum = torch.clamp(sum_p, min=1.0)
        chamfer = torch.sum(torch.abs(p - g) * (1.0 - g), dim=[-3, -2, -1]) / (p_sum * max(1, h * w))

        return (
            ssim.detach().cpu().numpy(),
            psnr.detach().cpu().numpy(),
            recall.detach().cpu().numpy(),
            f1.detach().cpu().numpy(),
            density.detach().cpu().numpy(),
            ghost_px.detach().cpu().numpy(),
            dark_ghost_px.detach().cpu().numpy(),
            chamfer.detach().cpu().numpy()
        )

print("[✓] Bộ đo lường batched GPU metrics sẵn sàng (SSIM, PSNR, Recall, F1, Density, Ghost, Chamfer).")
sys.stdout.flush()

In [ ]:
# [CELL 9]: Bộ Quản Lý Checkpoint Thông Minh V3.1 (Hard Gates Chuẩn Vàng 1.10x + Resume Checkpoint)
import os
import sys
import json
import torch

RESUME_CKPT_NAME = 'model_tps_v3.0_resume.pt'

class SmartCheckpointManager_v30:
    def __init__(self, save_dir, ref_ghost=9070.9, ref_chamfer=0.00230, ref_recall=0.930,
                 ref_ssim=0.900, ref_val_l1=0.0308, ref_dark_ghost=750.0,
                 min_recall=0.905, min_density=1.00, max_density=1.18, target_density=1.10):
        self.save_dir = save_dir
        self.ref_ghost = ref_ghost
        self.ref_chamfer = ref_chamfer
        self.ref_recall = ref_recall
        self.ref_ssim = ref_ssim
        self.ref_val_l1 = ref_val_l1
        self.ref_dark_ghost = ref_dark_ghost
        self.min_recall = min_recall
        self.min_density = min_density
        self.max_density = max_density
        self.target_density = target_density

        self.best_score = float('inf')
        self.best_progress_score = float('inf')
        self.best_epoch = 0
        self.resume_path = os.path.join(save_dir, RESUME_CKPT_NAME)
        os.makedirs(save_dir, exist_ok=True)

    def calculate_score(self, m):
        density_dev = abs(m['density'] - self.target_density)
        dark_ghost_term = m['dark_ghost'] / max(1.0, self.ref_dark_ghost)
        
        score = (
            0.25 * (m['ghost'] / self.ref_ghost) +
            0.20 * (m['chamfer'] / self.ref_chamfer) +
            0.15 * ((1.0 - m['recall']) / (1.0 - self.ref_recall)) +
            0.15 * ((1.0 - m['ssim']) / (1.0 - self.ref_ssim)) +
            0.15 * (m['val_l1'] / self.ref_val_l1) +
            0.05 * (density_dev / 0.10) +
            0.05 * dark_ghost_term
        )
        return float(score)

    @staticmethod
    def _atomic_save(obj, path):
        tmp = path + '.tmp'
        torch.save(obj, tmp)
        os.replace(tmp, path)

    def save_resume(self, epoch, model, optimizer=None, scaler=None, history=None, completed=True):
        """Lưu trạng thái đầy đủ để train tiếp. completed=False nghĩa là epoch `epoch` bị ngắt giữa chừng
        (watchdog) -> khi resume sẽ chạy lại chính epoch đó."""
        raw_model = model.module if hasattr(model, 'module') else (model.base_model if hasattr(model, 'base_model') else model)
        state = {
            'epoch': int(epoch) if completed else int(epoch) - 1,
            'model': {k: v.detach().cpu() for k, v in raw_model.state_dict().items()},
            'optimizer': optimizer.state_dict() if optimizer is not None else None,
            'scaler': scaler.state_dict() if scaler is not None else None,
            'best_score': self.best_score,
            'best_progress_score': self.best_progress_score,
            'best_epoch': self.best_epoch,
            'history': list(history) if history is not None else [],
        }
        self._atomic_save(state, self.resume_path)
        del state

    def load_resume(self, model, optimizer=None, scaler=None, map_location='cpu'):
        """Trả về (last_completed_epoch, history) hoặc (0, []) nếu không có resume."""
        if not os.path.exists(self.resume_path):
            return 0, []
        state = torch.load(self.resume_path, map_location=map_location, weights_only=False)
        raw_model = model.module if hasattr(model, 'module') else (model.base_model if hasattr(model, 'base_model') else model)
        raw_model.load_state_dict(state['model'], strict=False)
        if optimizer is not None and state.get('optimizer') is not None:
            try:
                optimizer.load_state_dict(state['optimizer'])
            except Exception as e:
                print(f"[!] Không nạp được optimizer state ({e}), dùng optimizer mới.")
        if scaler is not None and state.get('scaler') is not None:
            try:
                scaler.load_state_dict(state['scaler'])
            except Exception:
                pass
        self.best_score = state.get('best_score', float('inf'))
        self.best_progress_score = state.get('best_progress_score', float('inf'))
        self.best_epoch = state.get('best_epoch', 0)
        ep = int(state.get('epoch', 0))
        hist = [h for h in state.get('history', []) if int(h.get('epoch', 0)) <= ep]
        del state
        return ep, hist

    def evaluate_and_save(self, epoch, metrics, model, optimizer=None):
        score = self.calculate_score(metrics)
        raw_model = model.module if hasattr(model, 'module') else (model.base_model if hasattr(model, 'base_model') else model)
        sd = {k: v.detach().cpu() for k, v in raw_model.state_dict().items()}

        # 1. Lưu checkpoint latest
        latest_path = os.path.join(self.save_dir, 'model_tps_v3.0_latest.pt')
        self._atomic_save(sd, latest_path)

        # 2. Kiểm tra Hard Gates
        density_ok = (self.min_density <= metrics['density'] <= self.max_density)
        recall_ok = (metrics['recall'] >= self.min_recall)
        gates_passed = (density_ok and recall_ok)

        # 3. Cập nhật Best Checkpoint khi thỏa mãn Hard Gates
        is_best = False
        if gates_passed and (score < self.best_score):
            self.best_score = score
            self.best_epoch = epoch
            is_best = True
            best_path = os.path.join(self.save_dir, 'model_tps_v3.0_best.pt')
            self._atomic_save(sd, best_path)

        # 4. Cập nhật Best Progress
        is_progress = False
        if score < self.best_progress_score:
            self.best_progress_score = score
            is_progress = True
            prog_path = os.path.join(self.save_dir, 'model_tps_v3.0_best_progress.pt')
            self._atomic_save(sd, prog_path)

        del sd
        return score, gates_passed, is_best, is_progress

ckpt_manager = SmartCheckpointManager_v30(CKPT_DIR)
print(f"[✓] SmartCheckpointManager_v30 khởi tạo thành công tại: {CKPT_DIR}")
if os.path.exists(ckpt_manager.resume_path):
    print(f"[*] Phát hiện checkpoint resume: {ckpt_manager.resume_path} (Cell 10 sẽ tự train tiếp)")
sys.stdout.flush()


In [ ]:
# [CELL 10]: Vòng Lặp Huấn Luyện & Thẩm Định Hoàn Chỉnh (25 Epochs V3.0 - Zero Leak, Watchdog RAM & Resume)
import os
import sys
import time
import math
import cv2
import numpy as np
import torch
from tqdm.auto import tqdm
TOTAL_EPOCHS = 25
BATCH_SIZE = 8   # 4 mẫu/GPU trên 2x T4
GRAD_ACCUM_STEPS = 2
WATCHDOG_CHECK_EVERY = 100  # kiểm tra RAM mỗi N step train
history = []
train_start_time = time.time()
history_csv_path = os.path.join(WORKING_DIR, 'training_history_v3.0.csv')

# Thiết lập bộ tối ưu hóa AdamW 3 nhóm tham số
param_group_backbone = []
param_group_unet = []
param_group_v3 = []

for name, param in base_model.named_parameters():
    if not param.requires_grad:
        continue
    if any(k in name for k in ['asym_analyzer', 'contour_propagator', 'flow_propagator', 'asf_head']):
        param_group_v3.append(param)
    elif 'unet' in name or 'fusion' in name:
        param_group_unet.append(param)
    else:
        param_group_backbone.append(param)

optimizer = torch.optim.AdamW([
    {'params': param_group_backbone, 'lr': 0.0, 'weight_decay': 1e-4},
    {'params': param_group_unet, 'lr': 1.2e-5, 'weight_decay': 1e-4},
    {'params': param_group_v3, 'lr': 2.5e-5, 'weight_decay': 1e-4}
], betas=(0.9, 0.999), eps=1e-8)

scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

# [RESUME]: Tự động train tiếp nếu có checkpoint resume (sau watchdog / restart kernel)
start_epoch = 1
_last_ep, _hist = ckpt_manager.load_resume(base_model, optimizer, scaler)
if _last_ep > 0:
    start_epoch = _last_ep + 1
    history = _hist
    print(f"[✓] RESUME: Đã nạp trạng thái sau Epoch {_last_ep:02d} -> tiếp tục từ Epoch {start_epoch:02d} "
          f"(best_score={ckpt_manager.best_score:.4f})")
del _hist

def generate_checkpoint_visual_demo_v3(model, current_epoch, val_ds, dev, working_dir, num_samples=3):
    demo_dir = os.path.join(working_dir, 'visual_samples_v3.0')
    os.makedirs(demo_dir, exist_ok=True)
    if len(val_ds) == 0: return []
    step_idx = max(1, len(val_ds) // (num_samples + 1))
    sample_indices = [min(len(val_ds) - 1, step_idx * (i + 1)) for i in range(num_samples)]
    raw_model = model.module if hasattr(model, 'module') else (model.base_model if hasattr(model, 'base_model') else model)
    was_training = raw_model.training
    raw_model.eval()

    saved_files = []
    with torch.inference_mode():
        for s_idx, idx in enumerate(sample_indices):
            batch_single = custom_collate_fn_v3([val_ds[idx]])
            t0 = batch_single[0].to(dev)
            tgt = batch_single[1].to(dev)
            t1 = batch_single[2].to(dev)
            t_val = batch_single[3].to(dev)
            c0_b = batch_single[4].to(dev)
            th01_b = batch_single[5].to(dev)
            c1_b = batch_single[6].to(dev)
            th10_b = batch_single[7].to(dev)
            valid_b = batch_single[8].to(dev)
            clip_name = str(batch_single[9][0])
            orig_h = int(batch_single[10][0]) if len(batch_single) > 10 else 540
            orig_w = int(batch_single[11][0]) if len(batch_single) > 11 else 960
            del batch_single

            if t0.dtype == torch.uint8: t0 = ((255.0 - t0.float()) * (1.0 / 255.0)).contiguous()
            if tgt.dtype == torch.uint8: tgt = ((255.0 - tgt.float()) * (1.0 / 255.0)).contiguous()
            if t1.dtype == torch.uint8: t1 = ((255.0 - t1.float()) * (1.0 / 255.0)).contiguous()

            res = raw_model(t0, t1, t_val, tps_precomputed=(c0_b, th01_b, c1_b, th10_b, valid_b))
            pred_t, (v0, v1) = res[0], res[1]
            del res

            if orig_h > 0 and orig_w > 0 and (pred_t.shape[-2] != orig_h or pred_t.shape[-1] != orig_w):
                pred_t = pred_t[:, :, :orig_h, :orig_w]
                t0 = t0[:, :, :orig_h, :orig_w]
                tgt = tgt[:, :, :orig_h, :orig_w]
                v0 = v0[:, :, :orig_h, :orig_w]
                v1 = v1[:, :, :orig_h, :orig_w]

            img_0 = ((1.0 - t0) * 255.0).clamp(0, 255).byte().cpu().numpy()[0, 0]
            img_pred = ((1.0 - pred_t) * 255.0).clamp(0, 255).byte().cpu().numpy()[0, 0]
            img_gt = ((1.0 - tgt) * 255.0).clamp(0, 255).byte().cpu().numpy()[0, 0]
            v0_map = (v0.clamp(0, 1) * 255.0).byte().cpu().numpy()[0, 0]
            v1_map = (v1.clamp(0, 1) * 255.0).byte().cpu().numpy()[0, 0]
            vis_heatmap = np.stack([v0_map, np.zeros_like(v0_map), v1_map], axis=-1)

            p_bin = (pred_t > 0.15).float().sum().item()
            g_bin = (tgt > 0.15).float().sum().item()
            s_density = p_bin / max(1.0, g_bin)

            h_c, w_c = img_0.shape
            panel = np.hstack([
                cv2.cvtColor(img_0, cv2.COLOR_GRAY2BGR),
                vis_heatmap,
                cv2.cvtColor(img_pred, cv2.COLOR_GRAY2BGR),
                cv2.cvtColor(img_gt, cv2.COLOR_GRAY2BGR)
            ])
            cv2.putText(panel, "Frame 0", (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)
            cv2.putText(panel, "V0(B)/V1(R) Gates", (w_c + 15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(panel, f"V3.0 Ep{current_epoch:02d} (D:{s_density:.2f})", (2*w_c + 15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 150, 0), 2)
            cv2.putText(panel, "GT Target", (3*w_c + 15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)

            out_p = os.path.join(demo_dir, f"sample_ep{current_epoch:02d}_{s_idx+1:02d}_{clip_name}_v30.png")
            cv2.imwrite(out_p, panel)
            saved_files.append(out_p)
            del t0, tgt, t1, t_val, c0_b, th01_b, c1_b, th10_b, valid_b, pred_t, v0, v1, panel

    if was_training: raw_model.train()
    return saved_files

print("=" * 85)
print(f"🚀 BẮT ĐẦU HUẤN LUYỆN TINH CHỈNH TPS-INBETWEEN V3.0 ({TOTAL_EPOCHS} EPOCHS, bắt đầu từ Epoch {start_epoch:02d})")
print(f"   - Phân giải huấn luyện: 540p Toàn phần (Full-frame 544x960, Không crop) | Batch vật lý: {BATCH_SIZE} | Tích lũy: {GRAD_ACCUM_STEPS}")
print(f"   - Effective Batch Size : {BATCH_SIZE * GRAD_ACCUM_STEPS} | Workers: 4 (mmap 0-copy)")
print(f"   - RAM Container ban đầu: {format_host_ram()}")
print(f"   - Watchdog RAM: dừng êm & lưu resume khi Working Set > {RAM_SOFT_LIMIT_GB:.0f} GB")
print("=" * 85)
sys.stdout.flush()

watchdog_triggered = False

for epoch in range(start_epoch, TOTAL_EPOCHS + 1):
    t_ep_start = time.time()

    # Lịch trình 2 giai đoạn:
    print(f"\n{'=' * 85}")
    if epoch <= 2:
        for p in param_group_backbone: p.requires_grad = False
        lr_enc = 0.0
        lr_unet = 1.2e-5
        lr_v3 = 2.5e-5 if epoch == 1 else 5.0e-5
        print(f"[*] [GIAI ĐOẠN 1: EPOCH {epoch:02d} WARMUP]: Đóng băng Backbone V2.6, khởi động êm các nhánh V3.")
    else:
        for p in param_group_backbone: p.requires_grad = True
        progress = max(0.0, min(1.0, float(epoch - 3) / max(1.0, float(TOTAL_EPOCHS - 3))))
        cos_decay = 0.5 * (1.0 + math.cos(math.pi * progress))
        lr_enc = 1.0e-6 + (1.0e-5 - 1.0e-6) * cos_decay
        lr_unet = 1.5e-6 + (3.0e-5 - 1.5e-6) * cos_decay
        lr_v3 = 2.0e-6 + (6.0e-5 - 2.0e-6) * cos_decay
        print(f"[*] [GIAI ĐOẠN 2: EPOCH {epoch:02d} COSINE DECAY]: Toàn mạng mở khóa, cos_decay = {cos_decay:.4f}")
    print(f"{'=' * 85}")
    sys.stdout.flush()

    optimizer.param_groups[0]['lr'] = lr_enc
    optimizer.param_groups[1]['lr'] = lr_unet
    optimizer.param_groups[2]['lr'] = lr_v3

    # Cập nhật epoch cho Loss Curriculum
    if hasattr(parallel_engine, 'current_epoch'):
        parallel_engine.current_epoch = epoch
    elif hasattr(parallel_engine, 'module'):
        parallel_engine.module.current_epoch = epoch

    # -------------------------------------------------------------------------
    # TRAIN PHASE
    # -------------------------------------------------------------------------
    parallel_engine.train()
    running_train_loss = torch.tensor(0.0, device=device)
    running_l1 = torch.tensor(0.0, device=device)
    running_dens = torch.tensor(0.0, device=device)
    optimizer.zero_grad(set_to_none=True)

    n_train_steps = len(train_loader)
    train_iter = iter(train_loader)
    train_pbar = tqdm(train_iter, total=n_train_steps, desc=f"🏋️ [Ep {epoch:02d}/{TOTAL_EPOCHS:02d}] Train", unit="batch", leave=False)
    steps_done = 0
    for step, batch in enumerate(train_pbar):
        t0 = batch[0].to(device, non_blocking=True)
        tgt = batch[1].to(device, non_blocking=True)
        t1 = batch[2].to(device, non_blocking=True)
        t_val = batch[3].to(device, non_blocking=True)
        c0_b = batch[4].to(device, non_blocking=True)
        th01_b = batch[5].to(device, non_blocking=True)
        c1_b = batch[6].to(device, non_blocking=True)
        th10_b = batch[7].to(device, non_blocking=True)
        valid_b = batch[8].to(device, non_blocking=True)
        del batch

        res_losses = parallel_engine(t0, tgt, t1, t_val, c0_b, th01_b, c1_b, th10_b, valid_b, compute_loss=True)
        loss_val = res_losses[0].mean()
        loss = loss_val / GRAD_ACCUM_STEPS
        scaler.scale(loss).backward()

        # Tích lũy trực tiếp trên GPU dạng Tensor - KHÔNG gọi .item() mỗi step để giải phóng hoàn toàn CUDA queue!
        running_train_loss += loss_val.detach()
        running_l1 += res_losses[1].mean().detach()
        running_dens += res_losses[3].mean().detach()

        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == n_train_steps:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(base_model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        if step % 20 == 0 or (step + 1) == n_train_steps:
            curr_loss = float(loss_val.detach().item())
            curr_dens = float(res_losses[3].mean().detach().item())
            train_pbar.set_postfix({'Loss': f"{curr_loss:.4f}", 'DensL': f"{curr_dens:.3f}"})

        del t0, tgt, t1, t_val, c0_b, th01_b, c1_b, th10_b, valid_b, res_losses, loss, loss_val
        steps_done = step + 1

        # [WATCHDOG RAM]: kiểm tra working set container định kỳ
        if (step + 1) % WATCHDOG_CHECK_EVERY == 0:
            ws_now = get_host_ram_gb()
            if ws_now > RAM_SOFT_LIMIT_GB:
                watchdog_triggered = True
                print(f"\n🛑 [WATCHDOG]: Working Set = {ws_now:.2f} GB > {RAM_SOFT_LIMIT_GB:.0f} GB tại Ep {epoch:02d} step {step+1}/{n_train_steps}!")
                break

    train_pbar.close()
    del train_pbar, train_iter  # giải phóng iterator -> shutdown toàn bộ worker
    cleanup_memory()

    if watchdog_triggered:
        optimizer.zero_grad(set_to_none=True)
        ckpt_manager.save_resume(epoch, base_model, optimizer, scaler, history, completed=False)
        print(f"💾 Đã lưu checkpoint resume -> {ckpt_manager.resume_path} (sẽ chạy lại Epoch {epoch:02d}).")
        print(f"👉 Hãy xem hướng dẫn chi tiết ở ô Markdown ngay bên dưới để Restart Kernel và tiếp tục huấn luyện với RAM sạch.")
        print(f"   RAM hiện tại: {format_host_ram()}")
        sys.stdout.flush()
        break

    mean_train_loss = float(running_train_loss.item()) / max(1, steps_done)
    mean_train_l1 = float(running_l1.item()) / max(1, steps_done)

    # -------------------------------------------------------------------------
    # VALIDATION PHASE (THUẦN FP32 - CHỐNG BẪY ĐẢO MÀNG SƯƠNG)
    # -------------------------------------------------------------------------
    parallel_engine.eval()
    if hasattr(parallel_engine, 'use_amp'):
        parallel_engine.use_amp = False
    elif hasattr(parallel_engine, 'module'):
        parallel_engine.module.use_amp = False

    val_l1_accum = torch.tensor(0.0, device=device)
    val_ssim_list = []
    val_psnr_list = []
    val_recall_list = []
    val_f1_list = []
    val_dens_list = []
    val_ghost_list = []
    val_dark_ghost_list = []
    val_chamfer_list = []

    val_iter = iter(val_loader)
    val_pbar = tqdm(val_iter, total=len(val_loader), desc=f"🔍 [Ep {epoch:02d}/{TOTAL_EPOCHS:02d}] Val  ", unit="batch", leave=False)
    with torch.inference_mode():
        for step, batch in enumerate(val_pbar):
            t0 = batch[0].to(device, non_blocking=True)
            tgt = batch[1].to(device, non_blocking=True)
            t1 = batch[2].to(device, non_blocking=True)
            t_val = batch[3].to(device, non_blocking=True)
            c0_b = batch[4].to(device, non_blocking=True)
            th01_b = batch[5].to(device, non_blocking=True)
            c1_b = batch[6].to(device, non_blocking=True)
            th10_b = batch[7].to(device, non_blocking=True)
            valid_b = batch[8].to(device, non_blocking=True)
            orig_h = int(batch[10][0]) if len(batch) > 10 else 540
            orig_w = int(batch[11][0]) if len(batch) > 11 else 960
            del batch

            pred = parallel_engine(t0, tgt, t1, t_val, c0_b, th01_b, c1_b, th10_b, valid_b, compute_loss=False)

            if orig_h > 0 and orig_w > 0 and (pred.shape[-2] != orig_h or pred.shape[-1] != orig_w):
                pred = pred[:, :, :orig_h, :orig_w]
                tgt = tgt[:, :, :orig_h, :orig_w]

            if tgt.dtype == torch.uint8:
                tgt = ((255.0 - tgt.float()) * (1.0 / 255.0)).contiguous()

            v_l1 = F.l1_loss(pred.float(), tgt.float())
            val_l1_accum += v_l1.detach()

            m_ssim, m_psnr, m_rec, m_f1, m_dens, m_gh, m_dgh, m_ch = calculate_metrics_batch_gpu(pred, tgt)
            val_ssim_list.extend(m_ssim.tolist())
            val_psnr_list.extend(m_psnr.tolist())
            val_recall_list.extend(m_rec.tolist())
            val_f1_list.extend(m_f1.tolist())
            val_dens_list.extend(m_dens.tolist())
            val_ghost_list.extend(m_gh.tolist())
            val_dark_ghost_list.extend(m_dgh.tolist())
            val_chamfer_list.extend(m_ch.tolist())
            del t0, tgt, t1, t_val, c0_b, th01_b, c1_b, th10_b, valid_b, pred, v_l1

    val_pbar.close()
    del val_pbar, val_iter
    if hasattr(parallel_engine, 'use_amp'): parallel_engine.use_amp = True
    elif hasattr(parallel_engine, 'module'): parallel_engine.module.use_amp = True

    metrics_ep = {
        'val_l1': float(val_l1_accum.item()) / max(1, len(val_loader)),
        'ssim': float(np.mean(val_ssim_list)),
        'psnr': float(np.mean(val_psnr_list)),
        'recall': float(np.mean(val_recall_list)),
        'f1': float(np.mean(val_f1_list)),
        'density': float(np.mean(val_dens_list)),
        'ghost': float(np.mean(val_ghost_list)),
        'dark_ghost': float(np.mean(val_dark_ghost_list)),
        'chamfer': float(np.mean(val_chamfer_list))
    }

    score, gates_ok, is_best, is_progress = ckpt_manager.evaluate_and_save(epoch, metrics_ep, base_model, optimizer)

    if epoch % 5 == 0 or is_best or is_progress:
        demo_files = generate_checkpoint_visual_demo_v3(base_model, epoch, val_dataset, device, WORKING_DIR, num_samples=3)

    # Dọn TRƯỚC rồi mới đo -> log phản ánh đúng phần RAM bị giữ lại
    cleanup_memory()
    ep_time_m = (time.time() - t_ep_start) / 60.0
    mem = get_memory_stats()

    row = {
        'epoch': epoch,
        'train_loss': mean_train_loss,
        'val_l1': metrics_ep['val_l1'],
        'val_ssim': metrics_ep['ssim'],
        'val_psnr': metrics_ep['psnr'],
        'stroke_recall': metrics_ep['recall'],
        'stroke_f1': metrics_ep['f1'],
        'density_ratio': metrics_ep['density'],
        'ghost_pixels': metrics_ep['ghost'],
        'dark_ghost_pixels': metrics_ep['dark_ghost'],
        'composite_score': score,
        'host_ram_used_gb': mem['working_set'],
        'ram_anon_gb': mem['anon'],
        'ram_file_gb': mem['file'],
        'ram_shmem_gb': mem['shmem'],
        'ram_devshm_gb': mem['devshm'],
        'ram_pss_gb': mem['pss'],
        'time_minutes': ep_time_m
    }
    history.append(row)
    pd.DataFrame(history).to_csv(history_csv_path, index=False)

    # Lưu resume sau mỗi epoch hoàn chỉnh
    ckpt_manager.save_resume(epoch, base_model, optimizer, scaler, history, completed=True)

    print(f"\n📌 ĐÃ XONG EPOCH {epoch:02d}/{TOTAL_EPOCHS:02d} ({ep_time_m:.1f} phút)")
    print(f"🧠 RAM: {format_host_ram(mem)}")
    if mem['working_set'] > RAM_WARN_GB:
        print(f"⚠️ [CẢNH BÁO RAM]: Working Set {mem['working_set']:.2f} GB > {RAM_WARN_GB:.0f} GB (watchdog dừng ở {RAM_SOFT_LIMIT_GB:.0f} GB)")
    print(f"🔍 VAL: L1={metrics_ep['val_l1']:.4f} | SSIM={metrics_ep['ssim']:.4f} | PSNR={metrics_ep['psnr']:.2f}dB | Recall={metrics_ep['recall']*100:.1f}% | Density={metrics_ep['density']:.4f}x (Mục tiêu 1.10x)")
    print(f"🎯 Sakuga Composite Score V3.0: {score:.4f} (Kỷ lục: {ckpt_manager.best_score:.4f})")
    if not gates_ok:
        print(f"⚠️ [Hard Gates BỊ CHẶN]: Density={metrics_ep['density']:.4f} (chuẩn [1.00, 1.18]), Recall={metrics_ep['recall']*100:.1f}% (chuẩn >= 90.5%)")
    if is_best:
        print(f"🌟 [KỶ LỤC MỚI V3.0]: Checkpoint Đỉnh Cao Đã Lưu -> model_tps_v3.0_best.pt")
    elif is_progress:
        print(f"📈 [TIẾN TRÌNH TỐT NHẤT]: Lưu checkpoint tiến trình -> model_tps_v3.0_best_progress.pt")

    sys.stdout.flush()

if watchdog_triggered:
    print(f"\n⏸️ HUẤN LUYỆN TẠM DỪNG DO WATCHDOG RAM. Xem hướng dẫn ở ô Markdown ngay bên dưới để Restart Kernel và tiếp tục.")
else:
    print(f"\n🎉 HOÀN TẤT HUẤN LUYỆN TOÀN BỘ {TOTAL_EPOCHS} EPOCHS TRONG {(time.time() - train_start_time)/60:.1f} PHÚT!")
sys.stdout.flush()


### 🔄 HƯỚNG DẪN RESTART KERNEL & TIẾP TỤC HUẤN LUYỆN (KHI CHẠM NGƯỠNG RAM / WATCHDOG 28 GB)

> [!CAUTION]
> **TUYỆT ĐỐI KHÔNG BẤM NÚT "FACTORY RESET" (Biểu tượng xoay tròn ↺ ở góc trên bên phải thanh công cụ Kaggle):**  
> Nút Factory Reset sẽ xóa sạch máy ảo và làm mất toàn bộ thư mục `/kaggle/working` cùng các file checkpoint đã lưu!

Khi **Cell 10** chạm ngưỡng RAM và hiển thị thông báo `[WATCHDOG]: Working Set > 28 GB`, hãy làm theo 2 bước cực kỳ đơn giản sau:

---

#### Bước 1: Tạo Cell Code mới để Reset RAM về 1 GB sạch sẽ
Nhấn nút **`+ Code`** (ngay bên dưới Cell này), copy đoạn mã dưới đây, dán vào và bấm **Run (Play)**:

```python
import os
os._exit(0)
```

> **Cơ chế:** Lệnh `os._exit(0)` sẽ yêu cầu tiến trình Python kernel cũ thoát ngay lập tức. Hệ điều hành Linux sẽ thu hồi toàn bộ 28 GB RAM về mức sạch ban đầu (~1 GB). Sau 3 giây, Kaggle sẽ tự động khởi động một kernel mới. **Toàn bộ ổ đĩa `/kaggle/working` và các checkpoint được bảo toàn nguyên vẹn 100%!**

---

#### Bước 2: Nhấn "Run All" để tiếp tục huấn luyện
Sau khi kernel đã khởi động lại (thanh RAM Kaggle UI trở về màu xanh ~1.5 GB):
1. Nhấn nút **Run All** trên thanh công cụ notebook (hoặc chạy tuần tự từ Cell 1 đến Cell 10).
2. Các Cell 1 đến Cell 9 sẽ lướt qua trong 10–15 giây.
3. Đến **Cell 10**, hệ thống sẽ tự động phát hiện `model_tps_v3.0_resume.pt` và in log:
   ```text
   [✓] RESUME: Đã nạp trạng thái sau Epoch XX -> tiếp tục từ Epoch YY...
   ```
4. Quá trình huấn luyện sẽ tiếp tục mượt mà cho đến khi hoàn tất 25 epochs!


In [ ]:
# [CELL 11]: Trực Quan Hóa Tiến Trình Hội Tụ & Chỉ Số Huấn Luyện Qua Các Epoch V3.0
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

def plot_training_metrics_convergence_v30(history_csv=None, save_dir=None):
    if history_csv is None: history_csv = os.path.join(WORKING_DIR, 'training_history_v3.0.csv')
    if save_dir is None: save_dir = WORKING_DIR
    if not os.path.exists(history_csv):
        print(f"[!] Không tìm thấy file lịch sử {history_csv} để vẽ biểu đồ.")
        return

    df = pd.read_csv(history_csv)
    if len(df) == 0: return

    epochs = df['epoch'].values
    fig, axes = plt.subplots(2, 3, figsize=(22, 12), dpi=150)
    fig.suptitle('TPS-INBETWEEN V3.0 TRAINING CONVERGENCE & ANALYTICS (ZERO RAM LEAK & 540P SAFE)', fontsize=16, fontweight='bold')

    # 1. Loss Convergence
    ax = axes[0, 0]
    ax.plot(epochs, df['train_loss'], 'b-o', label='Train Total Loss', linewidth=2)
    ax.plot(epochs, df['val_l1'], 'r--s', label='Val L1 Loss', linewidth=2)
    ax.set_title('Loss Convergence (Train Total & Val L1)')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True)

    # 2. SSIM & PSNR
    ax = axes[0, 1]
    ax.plot(epochs, df['val_ssim'], 'g-^', label='Val SSIM', linewidth=2)
    ax2 = ax.twinx()
    ax2.plot(epochs, df['val_psnr'], 'm--d', label='Val PSNR (dB)', linewidth=2)
    ax.set_title('Structural Similarity (SSIM) & PSNR')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('SSIM', color='g')
    ax2.set_ylabel('PSNR (dB)', color='m')
    ax.grid(True)

    # 3. Stroke Recall & F1
    ax = axes[0, 2]
    ax.plot(epochs, df['stroke_recall'] * 100, 'c-o', label='Stroke Recall (%)', linewidth=2)
    ax.plot(epochs, df['stroke_f1'] * 100, 'y--v', label='Stroke F1 (%)', linewidth=2)
    ax.axhline(y=90.5, color='red', linestyle=':', label='Hard Gate Recall (90.5%)')
    ax.set_title('Stroke Detail Retention (Recall & F1)')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Percentage (%)')
    ax.legend()
    ax.grid(True)

    # 4. Stroke Density Ratio (Gold Band 1.10x)
    ax = axes[1, 0]
    ax.plot(epochs, df['density_ratio'], 'k-o', label='Density Ratio', linewidth=2)
    ax.axhspan(1.04, 1.14, color='green', alpha=0.20, label='Gold Visual Band [1.04, 1.14]')
    ax.axhline(y=1.10, color='green', linestyle='--', label='Target Gold 1.10x')
    ax.axhline(y=1.00, color='red', linestyle=':', label='Gate Min 1.00x')
    ax.axhline(y=1.18, color='red', linestyle=':', label='Gate Max 1.18x')
    ax.set_title('Stroke Density Ratio (Gold Standard 1.10x)')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Density Ratio')
    ax.legend(loc='upper right')
    ax.grid(True)

    # 5. Ghost Pixels
    ax = axes[1, 1]
    ax.plot(epochs, df['ghost_pixels'], 'orange', marker='o', label='Effective Ghost Pixels', linewidth=2)
    ax.plot(epochs, df['dark_ghost_pixels'], 'darkred', linestyle='--', marker='s', label='Dark Ghost Pixels', linewidth=2)
    ax.set_title('Artifact & Ghost Stroke Suppression')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Pixel Count')
    ax.legend()
    ax.grid(True)

    # 6. Host RAM Usage
    ax = axes[1, 2]
    if 'host_ram_used_gb' in df.columns:
        ax.plot(epochs, df['host_ram_used_gb'], 'purple', marker='s', linewidth=2.5, label='Host RAM (GB)')
        ax.axhline(y=30.0, color='red', linestyle='--', label='Kaggle RAM Ceiling (30 GB)')
        ax.set_ylim(0, 32)
        ax.set_title('Host RAM Stability (Zero Leak Across 25 Epochs)')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('RAM Used (GB)')
        ax.legend()
        ax.grid(True)

    plt.tight_layout()
    plot_p = os.path.join(save_dir, 'training_metrics_convergence_v30.png')
    plt.savefig(plot_p, bbox_inches='tight')
    plt.show()
    print(f"[✓] Đã xuất biểu đồ hội tụ V3.0 tại: {plot_p}")

plot_training_metrics_convergence_v30()

In [ ]:
# [CELL 12]: Kiểm Toán Trực Quan Định Tính & Bằng Chứng Nối Ngọn Tóc & Xoay Người V3.0
import os
import sys
import glob
import cv2
import matplotlib.pyplot as plt
import os
import glob
from IPython.display import display, Image

demo_dir = os.path.join(WORKING_DIR, 'visual_samples_v3.0')
sample_pngs = sorted(glob.glob(os.path.join(demo_dir, '*.png')))

if not sample_pngs:
    print(f"[*] Chưa có ảnh mẫu trong {demo_dir}.")
else:
    print(f"[*] Tìm thấy {len(sample_pngs)} ảnh kiểm toán trực quan V3.0:")
    for p in sample_pngs[-6:]:
        fn = os.path.basename(p)
        print(f"\n🖼️ Ảnh mẫu: {fn}")
        display(Image(filename=p, width=900))

In [ ]:
# [CELL 13]: Đóng Gói Thành Phẩm Sản Phẩm V3.0 & Xuất Link Tải Trực Tiếp
import os
import sys
import zipfile
from IPython.display import FileLink
import os
import shutil
import zipfile
from IPython.display import FileLink, display

zip_name = "TPS_Inbetween_v3.0_Final_Artifacts.zip"
zip_path = os.path.join(WORKING_DIR, zip_name)

print("=" * 80)
print(f"📦 ĐANG ĐÓNG GÓI THÀNH PHẨM V3.0: {zip_name}...")
print("=" * 80)

files_to_pack = []
best_pt = os.path.join(CKPT_DIR, 'model_tps_v3.0_best.pt')
prog_pt = os.path.join(CKPT_DIR, 'model_tps_v3.0_best_progress.pt')
latest_pt = os.path.join(CKPT_DIR, 'model_tps_v3.0_latest.pt')

for cp in [best_pt, prog_pt, latest_pt]:
    if os.path.exists(cp):
        files_to_pack.append((cp, os.path.join('checkpoints_v3.0', os.path.basename(cp))))

hist_csv = os.path.join(WORKING_DIR, 'training_history_v3.0.csv')
if os.path.exists(hist_csv):
    files_to_pack.append((hist_csv, 'training_history_v3.0.csv'))

plot_png = os.path.join(WORKING_DIR, 'training_metrics_convergence_v30.png')
if os.path.exists(plot_png):
    files_to_pack.append((plot_png, 'training_metrics_convergence_v30.png'))

demo_dir = os.path.join(WORKING_DIR, 'visual_samples_v3.0')
if os.path.exists(demo_dir):
    for fn in os.listdir(demo_dir):
        fp = os.path.join(demo_dir, fn)
        if os.path.isfile(fp):
            files_to_pack.append((fp, os.path.join('visual_samples_v3.0', fn)))

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for src, arc in files_to_pack:
        zf.write(src, arcname=arc)
        sz_mb = os.path.getsize(src) / (1024**2)
        print(f"   + Đã thêm: {arc} ({sz_mb:.2f} MB)")

zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print("=" * 80)
print(f"🎉 ĐÓNG GÓI HOÀN TẤT THÀNH CÔNG: {zip_path} ({zip_size_mb:.2f} MB)")
print("=" * 80)
print("\n👉 NHẤN VÀO ĐƯỜNG LINK DƯỚI ĐÂY ĐỂ TẢI TRỰC TIẾP TOÀN BỘ THÀNH PHẨM V3.0:")
display(FileLink(zip_name))